# Notebook Consistency Notice (read before running)

**Classification bug (fixed).** The original per-tool `normalize_*` functions mis-parsed
several raw tool outputs -- most notably KoAT parser crashes and MuVal's multi-line
output -- silently mapping them to `"unknown"` instead of a genuine tool error, and
MuVal incorrectly reused AProVE's first-non-empty-line rule instead of its own
last-non-empty-line rule. This was fixed by replacing the ad hoc `detect_tool_error` /
`normalize_*` logic with a single authoritative `classify(tool, raw, stored_result)`
function (see the classifier cell), and re-deriving every verdict from the
already-committed `raw_output` in `thesis_output/wikifunctions.db`.

**`verdict_corrected` is authoritative.** `tool_results` has two extra columns,
`verdict_corrected` and `failure_reason`, computed by running `classify()` over the
stored `raw_output` for every row in the analyzable set
(`translations.its_status='ok' AND c_status='ok'`). The old `result` / `raw_output`
columns are left untouched as a historical record -- do not aggregate `result` for any
new analysis; use `verdict_corrected`.

**Reproducible without Docker.** The Docker tool-execution cell starts with
`RUN_TOOLS = False`. With the default `False` it skips running the tools entirely and
reuses the `tool_results` / `raw_output` already committed in
`thesis_output/wikifunctions.db`. Set `RUN_TOOLS = True` (and have the Docker images
available) only if you actually want to re-run the tools from scratch.

**Intended run order** (skip to step 4 if you just want to reproduce the corrected
tables from the committed DB):
1. Setup / feature-extraction / translation cells (top of notebook) -- populates
   `implementations`, `features`, `translations`.
2. The classifier cell -- defines `classify()` and the `normalize_*` wrappers.
3. The Docker tool-execution cell -- with `RUN_TOOLS = True` only if you need fresh
   tool runs; otherwise leave `False` (it will just report it's reusing the committed
   DB and will not touch `raw_output`).
4. The embedded reclassification cell -- recomputes `verdict_corrected` /
   `failure_reason` from `raw_output`, no Docker required.
5. The corrected RQ2 / RQ3 / per-format / coverage / full-export cells. Older cells
   with the same purpose are marked "Superseded" and left as a historical record with
   an empty body; they are not part of the run path.


# Wikifunctions Termination Analysis Pipeline
**Master Thesis: Certainly Terminating Wikifunctions**  
Eddie Begovic — MSc Digital Economy, WU Vienna  
Supervisor: Univ.-Prof. Dr. Axel Polleres

---

## Overview

This notebook implements the full empirical pipeline for the thesis.

| Stage | What it does |
|-------|--------------|
| 1 | Extract Python implementations from the Wikifunctions XML dump |
| 2 | Classify each function into a structural category via AST analysis |
| 3 | Translate INT_LOOP functions to C and ITS format |
| 3b | Validate all C files with gcc before submitting to tools |
| 4 | Persist all results in a SQLite database |
| 5 | Run AProVE, KoAT, T2 via Docker |
| 6 | Aggregate results, answer RQ1–RQ3, export CSV |

---

## Analyzable fragment

| Supported | Not supported |
|-----------|---------------|
| Integer variables | Strings / lists / dicts |
| `while` loops | Float arithmetic |
| `for x in range(...)` | External / unknown calls |
| `if / else` | Recursion (tool-dependent) |
| `+`, `-`, `*`, `//`, `%`, `**` | Dynamic types, generators, I/O |
| Tuple assignments `x,y = y, x%y` | |

## Translation quality labels

| Label | Meaning | Used in tool comparison |
|-------|---------|------------------------|
| `exact` | Nothing skipped or abstracted | Yes — primary |
| `sound_overapprox` | Modulo abstracted or irrelevant statement skipped | Yes — with caveat |
| `rejected` | Unsafe approximation required | No — limitations only |
| `not_attempted` | Not in analyzable category | No |

## Key design decisions

**`.koat` files contain zero comments.** The AProVE/KoAT parser rejects semicolons.  
Metadata goes to a separate `.meta.txt` file per function.

**Modulo is abstracted in ITS.** `x % y` becomes a fresh variable `r` with  
guard `y > 0 && r >= 0 && r < y`. Mathematically sound for non-negative inputs.  
This is what allows GCD to prove terminating with ranking function `rank = y`.

**`tool_error` is separate from `unknown`.** AProVE's C frontend crashes on ARM64  
Docker with `NumberFormatException: rch64`. That is a tool bug, not a real MAYBE.

**ITS is the primary path, C is secondary.** ITS bypasses the broken C→LLVM  
frontend and maps directly to the ranking function theory in the thesis.

## Tool setup

- **AProVE** (ITS): `jckassing/aprove:1.0.0` — ARM64 Docker image from developers  
- **KoAT**: build from `aprove-developers/KoAT2-Releases` — AMD64 compatible  
- **T2**: build from `francoismichel/t2-docker-bionic64` — uses C files

---
## Stage 0 — Environment Setup

In [1]:
!pip install tqdm --quiet

You should consider upgrading via the 'd:\python3810\python.exe -m pip install --upgrade pip' command.


In [2]:
import ast
import bz2
import json
import re
import shutil
import sqlite3
import subprocess
import datetime
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Optional
from tqdm import tqdm

print("All imports successful.")

All imports successful.


In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Edit INPUT_DIR_HOST_* to match your actual folder paths.
# Docker will mount these directories into the containers.

DUMP_PATH          = Path("wikifunctionswiki-2026-05-01-p3p82100.xml")
OUTPUT_DIR         = Path("thesis_output")
DB_PATH            = OUTPUT_DIR / "wikifunctions.db"
C_OUT_DIR          = OUTPUT_DIR / "c_translations"
C_INVALID_DIR      = OUTPUT_DIR / "c_invalid"
ITS_OUT_DIR        = OUTPUT_DIR / "its_ari"
ITS_META_DIR       = OUTPUT_DIR / "its_meta"
TOOL_TIMEOUT       = 60

# REBUILD_FROM_SOURCE controls the whole from-scratch pipeline (XML extraction,
# feature extraction, translation, Stage 4 Persistence). True: re-derive the DB from
# the raw XML dump + Docker tools (needs both). False (default): reproduce all
# results from the already-committed wikifunctions.db -- no dump, no Docker needed.
REBUILD_FROM_SOURCE = False

# Docker volume mount paths
INPUT_DIR_HOST_ITS = str((OUTPUT_DIR / "its_ari").resolve())
INPUT_DIR_HOST_C   = str((OUTPUT_DIR / "c_translations").resolve())
T2_OUT_DIR         = OUTPUT_DIR / "t2_translations"
INPUT_DIR_HOST_T2  = str((OUTPUT_DIR / "t2_translations").resolve())

for d in [OUTPUT_DIR, C_OUT_DIR, C_INVALID_DIR, ITS_OUT_DIR, ITS_META_DIR, T2_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dump exists : {DUMP_PATH.exists()}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")

---
## Stage 1 — Extraction

Streams through the bz2-compressed XML dump one page at a time.  
Each page is a Wikifunctions ZObject encoded as JSON.  
We extract pages of type `Z14` (Implementation) where the language is `Z610` (Python).

In [4]:
@dataclass
class WFImplementation:
    zid:          str
    function_zid: str
    python_code:  str
    label:        str

def is_python(v) -> bool:
    if isinstance(v, str):  return v == "Z610"
    if isinstance(v, dict): return v.get("Z1K1") == "Z610" or v.get("Z6K1") == "Z610"
    return False

def get_function_ref(inner):
    ref = inner.get("Z14K1", "")
    if isinstance(ref, str):  return ref
    if isinstance(ref, dict): return ref.get("Z9K1", ref.get("Z6K1", ""))
    return ""

def get_label(zobj, fallback):
    try:
        for s in zobj.get("Z2K3", {}).get("Z12K1", []):
            if isinstance(s, dict) and s.get("Z11K1") == "Z1002":
                return s.get("Z11K2", fallback)
    except Exception: pass
    return fallback

def tag_local(elem):
    return elem.tag.split("}")[-1] if "}" in elem.tag else elem.tag

print("Helpers defined.")

Helpers defined.


In [5]:
def extract_python_implementations(dump_path):
    results, seen = [], set()
    cur_title = cur_text = None
    pages = found = 0
    print("Extracting...")
    open_fn = __import__("bz2").open if str(dump_path).endswith(".bz2") else open
    with open_fn(str(dump_path), "rb") as f:
        for event, elem in ET.iterparse(f, events=("end",)):
            local = tag_local(elem)
            if local == "title":  cur_title = elem.text or ""
            elif local == "text": cur_text  = elem.text or ""
            elif local == "page":
                pages += 1
                if pages % 5000 == 0:
                    print(f"  Pages: {pages:,} | Found: {found:,}")
                zid  = (cur_title or "").strip()
                text = (cur_text  or "").strip()
                if zid.startswith("Z") and text:
                    try:
                        zobj  = json.loads(text)
                        inner = zobj.get("Z2K2", zobj)
                        if isinstance(inner, dict):
                            zt = inner.get("Z1K1", "")
                            if isinstance(zt, dict): zt = zt.get("Z1K1", "")
                            if zt == "Z14":
                                cb = inner.get("Z14K3")
                                if isinstance(cb, dict):
                                    lang = cb.get("Z16K1", "")
                                    code = cb.get("Z16K2", "")
                                    if is_python(lang) and isinstance(code, str) and code.strip():
                                        if zid not in seen:
                                            seen.add(zid)
                                            results.append(WFImplementation(
                                                zid=zid,
                                                function_zid=get_function_ref(inner),
                                                python_code=code.strip(),
                                                label=get_label(zobj, zid),
                                            ))
                                            found += 1
                    except Exception: pass
                cur_title = cur_text = None
                elem.clear()
    print(f"\nDone. Pages: {pages:,} | Python implementations: {found:,}")
    return results

In [6]:
if REBUILD_FROM_SOURCE:
    implementations = extract_python_implementations(DUMP_PATH)
    print("\nSample (first 3):")
    for impl in implementations[:3]:
        print(f"  {impl.zid}  fn={impl.function_zid}  {impl.label!r}")
        print(f"    {impl.python_code[:80]!r}")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping XML extraction; reusing implementations already committed in the DB.")


REBUILD_FROM_SOURCE is False -- skipping XML extraction; reusing implementations already committed in the DB.


---
## Stage 2 — AST Categorization

Each function is parsed into an Abstract Syntax Tree **without executing it**.  
We record structural features, then assign a category.

| Category | What it means | Why it is handled this way |
|----------|---------------|----------------------------|
| `INT_LOOP_CLEAN` | Integer while/range loop, nothing else | Translated to C and ITS |
| `INT_LOOP_MIXED` | Integer loop + strings or lists also present | Translation attempted |
| `RECURSION_ONLY` | Calls itself, no loop | AProVE can handle but different format — out of scope |
| `ITER_LOOP` | Loops over a list or string | Loop count depends on collection length, not an integer bound |
| `NO_LOOPS` | No loop, no recursion | Trivially terminates — no tool needed |
| `UNSUPPORTED` | Float arithmetic only | Tools require integer arithmetic |
| `PARSE_ERROR` | Broken Python syntax | Cannot analyse |

In [7]:
@dataclass
class FunctionFeatures:
    zid:                 str
    parse_error:         bool = False
    parse_error_msg:     str  = ""
    has_while:           bool = False
    has_for_range:       bool = False
    has_for_iter:        bool = False
    has_break:           bool = False
    has_continue:        bool = False
    max_loop_depth:      int  = 0
    has_recursion:       bool = False
    has_int_arithmetic:  bool = False
    has_float:           bool = False
    has_string_ops:      bool = False
    has_list_ops:        bool = False
    has_dict_ops:        bool = False
    has_external_calls:  bool = False
    external_call_names: str  = ""
    num_functions:       int  = 0
    loc:                 int  = 0
    analysis_category:   str  = ""

print("FunctionFeatures defined.")

FunctionFeatures defined.


In [8]:
class FeatureVisitor(ast.NodeVisitor):
    def __init__(self, zid):
        self.f = FunctionFeatures(zid=zid)
        self._def:   set = set()
        self._calls: set = set()
        self._depth = 0

    def visit_FunctionDef(self, node):
        self.f.num_functions += 1; self._def.add(node.name); self.generic_visit(node)
    visit_AsyncFunctionDef = visit_FunctionDef

    def visit_While(self, node):
        self.f.has_while = True; self._depth += 1
        self.f.max_loop_depth = max(self.f.max_loop_depth, self._depth)
        self.generic_visit(node); self._depth -= 1

    def visit_For(self, node):
        self._depth += 1; self.f.max_loop_depth = max(self.f.max_loop_depth, self._depth)
        if (isinstance(node.iter, ast.Call) and isinstance(node.iter.func, ast.Name)
                and node.iter.func.id == "range"):
            self.f.has_for_range = True
        else:
            self.f.has_for_iter = True
        self.generic_visit(node); self._depth -= 1

    def visit_Break(self, node):    self.f.has_break    = True
    def visit_Continue(self, node): self.f.has_continue = True

    def visit_BinOp(self, node):
        int_ops = (ast.Add,ast.Sub,ast.Mult,ast.FloorDiv,ast.Mod,
                   ast.Pow,ast.LShift,ast.RShift,ast.BitAnd,ast.BitOr,ast.BitXor)
        if isinstance(node.op, int_ops): self.f.has_int_arithmetic = True
        if isinstance(node.op, ast.Div): self.f.has_float = True
        self.generic_visit(node)

    def visit_AugAssign(self, node):
        # FIX: augmented assignments (i += 1, n *= i) are integer arithmetic too.
        int_ops = (ast.Add,ast.Sub,ast.Mult,ast.FloorDiv,ast.Mod,
                   ast.Pow,ast.LShift,ast.RShift,ast.BitAnd,ast.BitOr,ast.BitXor)
        if isinstance(node.op, int_ops): self.f.has_int_arithmetic = True
        if isinstance(node.op, ast.Div): self.f.has_float = True
        self.generic_visit(node)

    def visit_Constant(self, node):
        if isinstance(node.value, float): self.f.has_float      = True
        if isinstance(node.value, str):   self.f.has_string_ops = True
        self.generic_visit(node)

    def visit_JoinedStr(self, node): self.f.has_string_ops = True; self.generic_visit(node)
    def visit_Subscript(self, node): self.f.has_list_ops   = True; self.generic_visit(node)
    def visit_Dict(self, node):      self.f.has_dict_ops   = True; self.generic_visit(node)
    def visit_List(self, node):      self.f.has_list_ops   = True; self.generic_visit(node)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name):
            n = node.func.id; self._calls.add(n)
            if n in ("str","chr","ord","len","format","join","split","strip","replace"):
                self.f.has_string_ops = True
            if n in ("list","append","extend","pop","sorted","reversed",
                     "zip","enumerate","map","filter"): self.f.has_list_ops = True
            if n in ("float","abs","round","sqrt","ceil","floor"): self.f.has_float = True
        elif isinstance(node.func, ast.Attribute):
            m = node.func.attr
            if m in ("upper","lower","strip","split","replace","startswith",
                     "endswith","format","join","encode","decode"): self.f.has_string_ops = True
            if m in ("append","extend","pop","insert","remove","sort",
                     "reverse","copy"): self.f.has_list_ops = True
        self.generic_visit(node)

    def finalise(self):
        self.f.has_recursion = bool(self._def & self._calls)
        known = {
            "range","int","float","str","bool","len","print","abs","min","max",
            "sum","list","dict","set","tuple","sorted","reversed","enumerate",
            "zip","map","filter","isinstance","type","hasattr","getattr","setattr",
            "append","pop","chr","ord","round","True","False","None","open","input",
        }
        ext = self._calls - self._def - known
        self.f.has_external_calls  = bool(ext)
        self.f.external_call_names = ",".join(sorted(ext))
        return self.f


def categorize_function(impl):
    feat = FunctionFeatures(zid=impl.zid, loc=impl.python_code.count("\n") + 1)
    try:
        tree = ast.parse(impl.python_code)
        v = FeatureVisitor(impl.zid); v.visit(tree)
        feat = v.finalise(); feat.loc = impl.python_code.count("\n") + 1
    except (SyntaxError, RecursionError) as e:
        feat.parse_error = True; feat.parse_error_msg = str(e)
    return feat


def classify_source(code):
    """Faithfulness-first categorization (6 categories). Only INT_LOOP_TRANSLATABLE
    is sent to the C/ITS/T2/KoAT translators; everything else is skipped. A function is
    INT_LOOP_TRANSLATABLE iff it can be FAITHFULLY translated by the start->loop
    translator: exactly one integer loop, straight-line body, no dropped constructs."""
    UNSUP = (ast.Pow, ast.LShift, ast.RShift, ast.BitAnd, ast.BitOr, ast.BitXor)
    try:
        t = ast.parse(code)
    except SyntaxError:
        return "PARSE_ERROR"
    _ds = set()
    for n in ast.walk(t):
        b = getattr(n, "body", None)
        if (isinstance(b, list) and b and isinstance(b[0], ast.Expr)
                and isinstance(b[0].value, ast.Constant)
                and isinstance(b[0].value.value, str)):
            _ds.add(id(b[0].value))
    loops   = [n for n in ast.walk(t) if isinstance(n, (ast.While, ast.For))]
    whiles  = [n for n in loops if isinstance(n, ast.While)]
    franges = [n for n in loops if isinstance(n, ast.For)
               and isinstance(n.iter, ast.Call)
               and isinstance(getattr(n.iter, "func", None), ast.Name)
               and n.iter.func.id == "range"]
    defs  = {n.name for n in ast.walk(t) if isinstance(n, ast.FunctionDef)}
    calls = {n.func.id for n in ast.walk(t)
             if isinstance(n, ast.Call) and isinstance(n.func, ast.Name)}
    has_recursion = bool(defs & calls)
    has_break     = any(isinstance(n, (ast.Break, ast.Continue)) for n in ast.walk(t))
    has_unsup     = any(isinstance(n, ast.BinOp) and isinstance(n.op, UNSUP) for n in ast.walk(t))
    non_integer = (
        any(isinstance(n, ast.Constant) and isinstance(n.value, (float, complex)) for n in ast.walk(t))
        or any(isinstance(n, ast.BinOp) and isinstance(n.op, ast.Div) for n in ast.walk(t))
        or any(isinstance(n, ast.Constant) and isinstance(n.value, str) and id(n) not in _ds for n in ast.walk(t))
        or any(isinstance(n, (ast.List, ast.Dict, ast.Set, ast.ListComp, ast.DictComp,
                              ast.SetComp, ast.Subscript, ast.JoinedStr)) for n in ast.walk(t))
        or bool(calls & {"str","chr","ord","float","sqrt","len","join","split",
                         "round","format","sorted","list","dict","set"})
    )
    n_loops = len(loops)
    if n_loops == 0 and not has_recursion:
        return "NO_LOOP"
    if has_recursion:
        return "RECURSION"
    if non_integer:
        return "NON_INTEGER_LOOP"
    one_loop = n_loops == 1
    straightline = one_loop and all(
        isinstance(st, (ast.Assign, ast.AugAssign))
        or (isinstance(st, ast.Expr) and isinstance(st.value, ast.Constant))
        or isinstance(st, ast.Pass)
        for st in loops[0].body)
    kind = "while" if whiles else ("for_range" if franges else "other")
    if (one_loop and straightline and kind in ("while", "for_range")
            and not has_break and not has_unsup):
        return "INT_LOOP_TRANSLATABLE"
    return "INT_LOOP_COMPLEX"


print("Categorization defined.")

Categorization defined.


In [9]:
if REBUILD_FROM_SOURCE:
    features_list = []
    for impl in tqdm(implementations, desc="AST analysis"):
        feat = categorize_function(impl)
        feat.analysis_category = classify_source(impl.python_code)
        features_list.append(feat)

    cat_counts = Counter(f.analysis_category for f in features_list)
    total = len(features_list)
    print("\n── Category distribution ──────────────────────────────")
    for cat, n in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f"  {cat:<22} {n:>5}  ({100*n/total:.1f}%)")
    print(f"  {'TOTAL':<22} {total:>5}")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping feature extraction; features already committed in the DB.")


REBUILD_FROM_SOURCE is False -- skipping feature extraction; features already committed in the DB.


In [10]:
if REBUILD_FROM_SOURCE:
    # ── LOOKUP STRUCTURES ─────────────────────────────────────────────────────────
    # This cell must always run — the category examples export and spot-check
    # cells both depend on by_cat and impl_by_zid.

    by_cat      = defaultdict(list)
    impl_by_zid = {i.zid: i for i in implementations}

    for feat in features_list:
        by_cat[feat.analysis_category].append(feat)

    print("by_cat and impl_by_zid ready.")
    for cat, items in sorted(by_cat.items(), key=lambda x: -len(x[1])):
        print(f"  {cat:<22} {len(items):>5}")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping (needs in-memory implementations/features_list).")


REBUILD_FROM_SOURCE is False -- skipping (needs in-memory implementations/features_list).


In [11]:
if REBUILD_FROM_SOURCE:
    # Quick peek: one example per category
    for cat in ["INT_LOOP_TRANSLATABLE","INT_LOOP_COMPLEX","NON_INTEGER_LOOP",
                "RECURSION","NO_LOOP","PARSE_ERROR"]:
        sample = by_cat.get(cat, [])
        if not sample: continue
        feat = sample[0]; impl = impl_by_zid.get(feat.zid)
        print(f"\n{'='*60}\nCategory: {cat}  ZID: {feat.zid}")
        if impl: print(impl.python_code[:250])
else:
    print("REBUILD_FROM_SOURCE is False -- skipping category peek (needs in-memory features).")


REBUILD_FROM_SOURCE is False -- skipping category peek (needs in-memory features).


In [12]:
if REBUILD_FROM_SOURCE:
    # Exports 2 real examples per category with full source + feature flags.
    examples_path = OUTPUT_DIR / "category_examples.txt"
    lines = []
    CATEGORY_REASONS = {
        "INT_LOOP_TRANSLATABLE": "INCLUDED - single integer loop, straight-line body; faithfully translatable to C, ITS, T2, KoAT.",
        "INT_LOOP_COMPLEX":      "EXCLUDED (for now) - integer loops but with in-loop conditionals, multiple/nested loops, break, or unsupported ops (**, <<, ...).",
        "NON_INTEGER_LOOP":      "EXCLUDED - loops operate on non-integer data (float, string, list, dict, date).",
        "RECURSION":             "EXCLUDED - uses recursion; out of scope for the loop-based translator.",
        "NO_LOOP":               "EXCLUDED - no loops and no recursion; trivially terminating.",
        "PARSE_ERROR":           "EXCLUDED - source does not parse (likely Python 2 syntax).",
    }
    for cat in ["INT_LOOP_TRANSLATABLE","INT_LOOP_COMPLEX","NON_INTEGER_LOOP",
                "RECURSION","NO_LOOP","PARSE_ERROR"]:
        samples = by_cat.get(cat, [])
        lines.append("=" * 70)
        lines.append(f"CATEGORY : {cat}")
        lines.append(f"COUNT    : {len(samples)} functions total")
        lines.append(f"REASON   : {CATEGORY_REASONS.get(cat, '')}")
        lines.append("=" * 70)
        if not samples:
            lines.append("  (no functions in this category)"); lines.append(""); continue
        for feat in samples[:2]:
            impl = impl_by_zid.get(feat.zid)
            if not impl: continue
            lines.append("")
            lines.append(f"  ZID    : {feat.zid}")
            lines.append(f"  Label  : {impl.label}")
            lines.append(f"  LOC    : {feat.loc}")
            lines.append("  Features:")
            lines.append(f"    while={feat.has_while}  for_range={feat.has_for_range}  "
                         f"for_iter={feat.has_for_iter}  recursion={feat.has_recursion}")
            lines.append(f"    int_arithmetic={feat.has_int_arithmetic}  "
                         f"strings={feat.has_string_ops}  lists={feat.has_list_ops}  float={feat.has_float}")
            if feat.parse_error:
                lines.append(f"    parse_error: {feat.parse_error_msg}")
            lines.append("  Python source:")
            for src_line in impl.python_code.split("\n"):
                lines.append(f"    {src_line}")
            lines.append("")
    output_text = "\n".join(lines)
    examples_path.write_text(output_text, encoding="utf-8")
    print(f"Exported: {examples_path}")
    print(output_text[:2000])
else:
    print("REBUILD_FROM_SOURCE is False -- skipping category_examples.txt export (needs in-memory features).")


REBUILD_FROM_SOURCE is False -- skipping category_examples.txt export (needs in-memory features).


---
## Stage 3 — Translation

Functions in `INT_LOOP_CLEAN` and `INT_LOOP_MIXED` are translated to two formats.

### C translation
- Parameters become `int` arguments
- Local variables get `int var = 0` declarations
- Tuple assignments use temporaries: `x,y = y, x%y` → `_t0=y; _t1=x%y; x=_t0; y=_t1`
- If `%` or `//` used: `__VERIFIER_assume(param >= 0)` added for each param
- `range(10, 0, -1)` generates `i > 0` not `i < 0` (direction-sensitive)
- gcc validates every file before it reaches a tool

### ITS translation
- Separate expression converter from C (ITS uses `=` not `==`, no `abs/min/max`, no `pow()`)
- Modulo abstraction: `x % y` → fresh variable `r` with `y > 0 && r >= 0 && r < y`
- Precondition on start transition guard (not just a comment)
- Zero comments in `.koat` files — parser rejects them
- All metadata written to `.meta.txt` files

### Output formats

| Format | Folder | Used by | Notes |
|--------|--------|---------|-------|
| `.c` | `c_translations/` | AProVE, MuVal | C Programs category |
| `.ari` | `its_ari/` | — | termCOMP 2025 standard; Docker images predate ARI support |
| `.koat` | `its_translations/` | KoAT | Legacy; KoAT2 rejects arithmetic in RHS, ~5/30 files parseable |
| `.t2` | `t2_translations/` | T2, VeryMax | Legacy format required by both tools |


In [13]:
@dataclass
class TranslationResult:
    zid:                   str
    c_code:                Optional[str] = None
    c_status:              str           = "not_attempted"
    c_fail_reason:         str           = ""
    c_approximations:      int           = 0
    c_quality:             str           = "not_attempted"
    c_has_precond:         bool          = False
    its_code:              Optional[str] = None
    its_status:            str           = "not_attempted"
    its_fail_reason:       str           = ""
    its_approximations:    int           = 0
    its_quality:           str           = "not_attempted"
    its_has_precond:       bool          = False
    its_modulo_abstracted: bool          = False
    t2_code:               Optional[str] = None
    t2_status:             str           = "not_attempted"
    t2_fail_reason:        str           = ""

print("TranslationResult defined.")

TranslationResult defined.


In [14]:
# ── SHARED UTILITIES ───────────────────────────────────────────────────────────

class CTranslationError(Exception):
    pass


def normalise_condition(cond: str) -> str:
    """while(y) → while(y != 0) for unambiguous tool parsing."""
    if re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', cond.strip()):
        return f"{cond.strip()} != 0"
    return cond


def uses_modulo(fn_node) -> bool:
    return any(
        isinstance(n, ast.BinOp) and isinstance(n.op, (ast.Mod, ast.FloorDiv))
        for n in ast.walk(fn_node)
    )


def loop_body_has_modulo(stmts) -> bool:
    for stmt in stmts:
        if any(isinstance(n, ast.BinOp) and isinstance(n.op, (ast.Mod, ast.FloorDiv))
               for n in ast.walk(stmt)):
            return True
    return False


def find_first_while(fn):
    for s in fn.body:
        if isinstance(s, ast.While): return s
    for s in fn.body:
        if isinstance(s, ast.If):
            for inner in s.body + s.orelse:
                if isinstance(inner, ast.While): return inner
    for node in ast.walk(fn):
        if isinstance(node, ast.While): return node
    return None


def collect_local_vars(fn_node) -> list:
    params = {a.arg for a in fn_node.args.args}
    assigned = []
    for node in ast.walk(fn_node):
        if isinstance(node, ast.Assign):
            tgt = node.targets[0] if node.targets else None
            if isinstance(tgt, ast.Name):
                v = tgt.id
                if v not in params and v not in assigned: assigned.append(v)
            elif isinstance(tgt, ast.Tuple):
                for elt in tgt.elts:
                    if isinstance(elt, ast.Name):
                        v = elt.id
                        if v not in params and v not in assigned: assigned.append(v)
        elif isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name):
            v = node.target.id
            if v not in params and v not in assigned: assigned.append(v)
        elif isinstance(node, ast.For) and isinstance(node.target, ast.Name):
            v = node.target.id
            if v not in params and v not in assigned: assigned.append(v)
    return assigned


def names_read(node) -> set:
    return {n.id for n in ast.walk(node)
            if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load)}


def names_written(node) -> set:
    return {n.id for n in ast.walk(node)
            if isinstance(n, ast.Name) and isinstance(n.ctx, (ast.Store, ast.Del))}


def side_effect_free_ignorable(stmt) -> bool:
    if isinstance(stmt, ast.Pass): return True
    if isinstance(stmt, ast.Expr): return isinstance(stmt.value, ast.Constant)
    return False


def range_parts(args, expr_conv) -> tuple:
    """Returns (start, end, step, direction). Symbolic steps are rejected."""
    if len(args) == 1:
        return "0", expr_conv(args[0]), "1", "<"
    if len(args) == 2:
        return expr_conv(args[0]), expr_conv(args[1]), "1", "<"
    if len(args) == 3:
        sn = args[2]; s, e, st = expr_conv(args[0]), expr_conv(args[1]), expr_conv(sn)
        val = None
        if isinstance(sn, ast.Constant) and isinstance(sn.value, int):
            val = sn.value
        elif (isinstance(sn, ast.UnaryOp) and isinstance(sn.op, ast.USub)
              and isinstance(sn.operand, ast.Constant)
              and isinstance(sn.operand.value, int)):
            val = -sn.operand.value
        if val is None:
            raise CTranslationError("range() symbolic step: cannot determine direction")
        if val == 0:
            raise CTranslationError("range() step 0 is invalid")
        return s, e, st, ("<" if val > 0 else ">")
    raise CTranslationError("range() unsupported argument count")


def precondition_for_vars(var_names: list) -> tuple:
    """Returns (its_guard_str, c_assume_lines_str) for non-negativity precondition."""
    if not var_names: return "TRUE", ""
    its = " && ".join(f"{v} >= 0" for v in var_names)
    c   = "\n".join(f"  __VERIFIER_assume({v} >= 0);" for v in var_names)
    return its, c


def finalize_quality(approximations: int) -> str:
    return "exact" if approximations == 0 else "sound_overapprox"


print("Shared utilities defined.")

Shared utilities defined.


In [15]:
# ── C EXPRESSION CONVERTER ────────────────────────────────────────────────────

def python_expr_to_c(node) -> str:
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool):  return "1" if node.value else "0"
        if isinstance(node.value, int):   return str(node.value)
        if isinstance(node.value, float): return str(int(node.value))
        raise CTranslationError(f"Non-numeric constant: {node.value!r}")
    if isinstance(node, ast.BinOp):
        ops = {ast.Add:"+",ast.Sub:"-",ast.Mult:"*",ast.FloorDiv:"/",ast.Mod:"%",ast.Div:"/"}
        if isinstance(node.op, ast.Pow):
            return f"(int)pow({python_expr_to_c(node.left)}, {python_expr_to_c(node.right)})"
        if type(node.op) not in ops:
            raise CTranslationError(f"Unsupported binary op: {type(node.op).__name__}")
        return f"({python_expr_to_c(node.left)} {ops[type(node.op)]} {python_expr_to_c(node.right)})"
    if isinstance(node, ast.UnaryOp):
        if isinstance(node.op, ast.USub): return f"(-{python_expr_to_c(node.operand)})"
        if isinstance(node.op, ast.UAdd): return python_expr_to_c(node.operand)
        if isinstance(node.op, ast.Not):  return f"(!{python_expr_to_c(node.operand)})"
        raise CTranslationError(f"Unsupported unary op: {type(node.op).__name__}")
    if isinstance(node, ast.Compare):
        cmp = {ast.Lt:"<",ast.LtE:"<=",ast.Gt:">",ast.GtE:">=",ast.Eq:"==",ast.NotEq:"!="}
        if len(node.ops) != 1: raise CTranslationError("Chained comparisons not supported")
        if type(node.ops[0]) not in cmp:
            raise CTranslationError(f"Unsupported comparison: {type(node.ops[0]).__name__}")
        return (f"{python_expr_to_c(node.left)} {cmp[type(node.ops[0])]} "
                f"{python_expr_to_c(node.comparators[0])}")
    if isinstance(node, ast.BoolOp):
        op = "&&" if isinstance(node.op, ast.And) else "||"
        return f" {op} ".join(f"({python_expr_to_c(v)})" for v in node.values)
    if isinstance(node, ast.IfExp):
        return (f"({python_expr_to_c(node.test)} ? "
                f"{python_expr_to_c(node.body)} : {python_expr_to_c(node.orelse)})")
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        n = node.func.id
        if n == "abs" and len(node.args)==1: return f"abs({python_expr_to_c(node.args[0])})"
        if n == "int" and len(node.args)==1: return python_expr_to_c(node.args[0])
        if n == "min" and len(node.args)==2:
            a,b = python_expr_to_c(node.args[0]),python_expr_to_c(node.args[1])
            return f"(({a})<({b})?({a}):({b}))"
        if n == "max" and len(node.args)==2:
            a,b = python_expr_to_c(node.args[0]),python_expr_to_c(node.args[1])
            return f"(({a})>({b})?({a}):({b}))"
    raise CTranslationError(f"Unsupported C expression: {type(node).__name__}")


print("C expression converter defined.")

C expression converter defined.


In [16]:
# ── ITS EXPRESSION CONVERTER ──────────────────────────────────────────────────
# Completely separate from C — KoAT ITS format has different syntax rules.
# Equality: = not ==
# Pow: expanded for small integer literals only
# abs/min/max/not: hard fail (not valid ITS arithmetic)

def python_expr_to_its(node) -> str:
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool):  return "1" if node.value else "0"
        if isinstance(node.value, int):   return str(node.value)
        if isinstance(node.value, float): return str(int(node.value))
        raise CTranslationError(f"Non-numeric constant in ITS: {node.value!r}")
    if isinstance(node, ast.BinOp):
        ops = {ast.Add:"+",ast.Sub:"-",ast.Mult:"*",ast.FloorDiv:"/",ast.Mod:"%",ast.Div:"/"}
        if isinstance(node.op, ast.Pow):
            base = python_expr_to_its(node.left)
            if isinstance(node.right, ast.Constant) and isinstance(node.right.value, int):
                exp = node.right.value
                if exp == 0: return "1"
                if exp == 1: return base
                if 2 <= exp <= 4: return "(" + " * ".join([f"({base})"]*exp) + ")"
            raise CTranslationError("Non-trivial Pow not supported in ITS")
        if type(node.op) not in ops:
            raise CTranslationError(f"Unsupported binary op in ITS: {type(node.op).__name__}")
        return (f"({python_expr_to_its(node.left)} {ops[type(node.op)]} "
                f"{python_expr_to_its(node.right)})")
    if isinstance(node, ast.UnaryOp):
        if isinstance(node.op, ast.USub): return f"(-{python_expr_to_its(node.operand)})"
        if isinstance(node.op, ast.UAdd): return python_expr_to_its(node.operand)
        if isinstance(node.op, ast.Not):
            raise CTranslationError("'not' not supported in ITS")
        raise CTranslationError(f"Unsupported unary op in ITS: {type(node.op).__name__}")
    if isinstance(node, ast.Compare):
        cmp = {ast.Lt:"<",ast.LtE:"<=",ast.Gt:">",ast.GtE:">=",
               ast.Eq:"=",ast.NotEq:"!="}   # ITS uses = not ==
        if len(node.ops) != 1: raise CTranslationError("Chained comparisons not supported in ITS")
        if type(node.ops[0]) not in cmp:
            raise CTranslationError(f"Unsupported comparison in ITS: {type(node.ops[0]).__name__}")
        return (f"{python_expr_to_its(node.left)} {cmp[type(node.ops[0])]} "
                f"{python_expr_to_its(node.comparators[0])}")
    if isinstance(node, ast.BoolOp):
        op = "&&" if isinstance(node.op, ast.And) else "||"
        return f" {op} ".join(f"({python_expr_to_its(v)})" for v in node.values)
    if isinstance(node, ast.IfExp):
        raise CTranslationError("Ternary not supported in ITS")
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        n = node.func.id
        if n == "int" and len(node.args)==1: return python_expr_to_its(node.args[0])
        if n in ("abs","min","max"):
            raise CTranslationError(f"{n}() not valid ITS arithmetic")
    raise CTranslationError(f"Unsupported ITS expression: {type(node).__name__}")


print("ITS expression converter defined.")

ITS expression converter defined.

In [17]:
# ── TUPLE ASSIGNMENT (C only) ─────────────────────────────────────────────────
# Python: x, y = y, x % y
# C:      { int _t0=y; int _t1=(x%y); x=_t0; y=_t1; }
# Temporaries preserve simultaneous evaluation.
# ITS does NOT need this — ITS evaluates all RHS from pre-state by definition.

def translate_tuple_assign_c(stmt) -> str:
    tgts = stmt.targets[0]; vals = stmt.value
    if not isinstance(tgts, ast.Tuple): raise CTranslationError("Not a tuple target")
    if not isinstance(vals, ast.Tuple): raise CTranslationError("Tuple needs tuple value")
    if len(tgts.elts) != len(vals.elts): raise CTranslationError("Tuple length mismatch")
    lines, tmps = [], []
    for i, rhs in enumerate(vals.elts):
        tmp = f"_t{i}"; tmps.append(tmp)
        lines.append(f"int {tmp} = {python_expr_to_c(rhs)};")
    for i, lhs in enumerate(tgts.elts):
        if not isinstance(lhs, ast.Name): raise CTranslationError("Non-name in tuple target")
        lines.append(f"{lhs.id} = {tmps[i]};")
    return "{\n      " + "\n      ".join(lines) + "\n    }"


print("Tuple assignment translator defined.")

Tuple assignment translator defined.


In [18]:
# ── C STATEMENT CONVERTER ─────────────────────────────────────────────────────

def python_stmt_to_c(stmt, indent: int = 4) -> str:
    pad  = " " * indent
    pad2 = " " * (indent + 2)
    if isinstance(stmt, ast.Assign):
        if not stmt.targets: raise CTranslationError("Assignment with no targets")
        tgt = stmt.targets[0]
        if isinstance(tgt, ast.Tuple): return translate_tuple_assign_c(stmt)
        if not isinstance(tgt, ast.Name): raise CTranslationError("Non-name assignment target")
        return f"{tgt.id} = {python_expr_to_c(stmt.value)};"
    if isinstance(stmt, ast.AugAssign):
        aug = {ast.Add:"+=",ast.Sub:"-=",ast.Mult:"*=",
               ast.FloorDiv:"/=",ast.Mod:"%=",ast.Div:"/="}
        if not isinstance(stmt.target, ast.Name):
            raise CTranslationError("Non-name aug-assign target")
        if type(stmt.op) not in aug:
            raise CTranslationError(f"Unsupported aug-assign: {type(stmt.op).__name__}")
        return f"{stmt.target.id} {aug[type(stmt.op)]} {python_expr_to_c(stmt.value)};"
    if isinstance(stmt, ast.Return):
        if stmt.value is None: return "return 0;"
        return f"return {python_expr_to_c(stmt.value)};"
    if isinstance(stmt, ast.If):
        cond  = normalise_condition(python_expr_to_c(stmt.test))
        then_ = "\n".join(pad2 + python_stmt_to_c(s, indent+2) for s in stmt.body)
        if stmt.orelse:
            else_ = "\n".join(pad2 + python_stmt_to_c(s, indent+2) for s in stmt.orelse)
            return f"if ({cond}) {{\n{then_}\n{pad}}} else {{\n{else_}\n{pad}}}"
        return f"if ({cond}) {{\n{then_}\n{pad}}}"
    if isinstance(stmt, ast.While):
        cond  = normalise_condition(python_expr_to_c(stmt.test))
        body_ = "\n".join(pad2 + python_stmt_to_c(s, indent+2) for s in stmt.body)
        return f"while ({cond}) {{\n{body_}\n{pad}}}"
    if isinstance(stmt, ast.For):
        if (isinstance(stmt.iter, ast.Call)
                and isinstance(stmt.iter.func, ast.Name)
                and stmt.iter.func.id == "range"
                and isinstance(stmt.target, ast.Name)):
            lv = stmt.target.id
            s_c, e_c, st_c, direction = range_parts(stmt.iter.args, python_expr_to_c)
            body_ = "\n".join(pad2 + python_stmt_to_c(s, indent+2) for s in stmt.body)
            return (f"for (int {lv} = {s_c}; {lv} {direction} {e_c}; "
                    f"{lv} += {st_c}) {{\n{body_}\n{pad}}}")
        raise CTranslationError("for loop over non-range iterator")
    if isinstance(stmt, ast.Expr):
        if side_effect_free_ignorable(stmt):
            return "/* skipped side-effect-free expression */;"
        raise CTranslationError("standalone expression with possible side effects")
    if isinstance(stmt, ast.Pass): return "/* pass */;"
    if isinstance(stmt, (ast.Break, ast.Continue)):
        raise CTranslationError(f"Control-flow not supported: {type(stmt).__name__}")
    raise CTranslationError(f"Unsupported statement: {type(stmt).__name__}")


def try_stmt_c(stmt, loop_relevant_vars=None, indent=4) -> tuple:
    """Returns (c_code, approx_count).
    Only silently skips statements that are provably side-effect-free
    AND do not write any loop-relevant variable.
    Anything else raises and causes a rejected translation.
    """
    try:
        return python_stmt_to_c(stmt, indent), 0
    except CTranslationError as e:
        lrv = set(loop_relevant_vars or [])
        if side_effect_free_ignorable(stmt) and not (names_written(stmt) & lrv):
            src = ast.unparse(stmt) if hasattr(ast, "unparse") else "<stmt>"
            return f"/* SOUND_APPROX skipped: {src[:80]} */;", 1
        src = ast.unparse(stmt) if hasattr(ast, "unparse") else "<stmt>"
        raise CTranslationError(f"unsafe_approximation: {e}; stmt={src[:120]}")


print("C statement converter defined.")

C statement converter defined.


In [19]:
# ── PYTHON → C TRANSLATOR ──────────────────────────────────────────────────────

def translate_to_c(impl, feat) -> TranslationResult:
    result = TranslationResult(zid=impl.zid)
    try:
        tree    = ast.parse(impl.python_code)
        fn_defs = [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
        if not fn_defs: raise CTranslationError("No function definition")
        fn = fn_defs[0]; name = fn.name
        params   = [a.arg for a in fn.args.args]
        params_c = ", ".join(f"int {p}" for p in params) or "void"
        approx   = 0

        has_mod = uses_modulo(fn)
        result.c_has_precond = has_mod
        _, c_assumes = precondition_for_vars(params) if has_mod else ("", "")
        verifier_decl = "extern void __VERIFIER_assume(int);\n"
        assume_block  = ("\n" + c_assumes) if c_assumes else ""
        precond_cmt   = (
            "// PRECONDITION: inputs >= 0 (via __VERIFIER_assume below)\n"
            "// Ensures Python % and C % agree (differ for negative operands)\n"
            if has_mod else ""
        )
        local_vars = collect_local_vars(fn)
        decl_block = ("\n" + "\n".join(f"  int {v} = 0;" for v in local_vars)) if local_vars else ""

        # ── Case 1: while loop ─────────────────────────────────────────────────
        wn = find_first_while(fn)
        if wn:
            cond         = normalise_condition(python_expr_to_c(wn.test))
            loop_relevant = names_read(wn.test) | names_written(wn)
            pre_stmts = []
            for stmt in fn.body:
                if stmt is wn: break
                if isinstance(stmt,(ast.Assign,ast.AugAssign,ast.Return,ast.If,ast.Expr,ast.Pass)):
                    c_str, ap = try_stmt_c(stmt, loop_relevant)
                    pre_stmts.append("  "+c_str); approx += ap
            pre_c = ("\n"+"\n".join(pre_stmts)) if pre_stmts else ""
            body_lines = []
            for s in wn.body:
                c_str, ap = try_stmt_c(s, loop_relevant)
                body_lines.append("    "+c_str); approx += ap
            nondet_decl   = "extern int __VERIFIER_nondet_int(void);\n"
            param_assigns = "\n".join(f"  int {p} = __VERIFIER_nondet_int();" for p in params)
            assume_main   = ("\n" + "\n".join(f"  __VERIFIER_assume({p} >= 0);" for p in params)) if has_mod else ""
            call_args     = ", ".join(params) if params else ""
            main_fn       = (
                f"\nint main() {{\n"
                f"{param_assigns}{assume_main}\n"
                f"  {name}({call_args});\n"
                f"  return 0;\n}}\n"
            )
            result.c_code = (
                f"{nondet_decl}{verifier_decl}"
                f"// Wikifunctions {impl.zid} — {impl.label}\n{precond_cmt}"
                f"// Translation quality: {finalize_quality(approx)}\n"
                f"int {name}({params_c}) {{{decl_block}{assume_block}{pre_c}\n"
                f"  while ({cond}) {{\n"
                f"{chr(10).join(body_lines)}\n  }}\n  return 0;\n}}\n"
                f"{main_fn}"
            )
            result.c_status = "ok"; result.c_approximations = approx
            result.c_quality = finalize_quality(approx); return result

        # ── Case 2: for x in range(...) ────────────────────────────────────────
        for_loops = [s for s in ast.walk(fn)
                     if isinstance(s,ast.For) and isinstance(s.iter,ast.Call)
                     and isinstance(s.iter.func,ast.Name) and s.iter.func.id=="range"]
        if for_loops:
            fl = for_loops[0]
            if not isinstance(fl.target, ast.Name):
                raise CTranslationError("range target not a simple variable")
            lv = fl.target.id
            s_c, e_c, st_c, direction = range_parts(fl.iter.args, python_expr_to_c)
            loop_relevant = {lv} | names_written(fl) | set(params)
            pre_stmts = []
            for stmt in fn.body:
                if stmt is fl: break
                if isinstance(stmt,(ast.Assign,ast.AugAssign,ast.Return,ast.If,ast.Expr,ast.Pass)):
                    c_str, ap = try_stmt_c(stmt, loop_relevant)
                    pre_stmts.append("  "+c_str); approx += ap
            pre_c = ("\n"+"\n".join(pre_stmts)) if pre_stmts else ""
            body_lines = []
            for s in fl.body:
                c_str, ap = try_stmt_c(s, loop_relevant)
                body_lines.append("    "+c_str); approx += ap
            nondet_decl   = "extern int __VERIFIER_nondet_int(void);\n"
            param_assigns = "\n".join(f"  int {p} = __VERIFIER_nondet_int();" for p in params)
            assume_main   = ("\n" + "\n".join(f"  __VERIFIER_assume({p} >= 0);" for p in params)) if has_mod else ""
            call_args     = ", ".join(params) if params else ""
            main_fn       = (
                f"\nint main() {{\n"
                f"{param_assigns}{assume_main}\n"
                f"  {name}({call_args});\n"
                f"  return 0;\n}}\n"
            )
            result.c_code = (
                f"{nondet_decl}{verifier_decl}"
                f"// Wikifunctions {impl.zid} — {impl.label}\n{precond_cmt}"
                f"// for {lv} in range (direction {direction})\n"
                f"// Translation quality: {finalize_quality(approx)}\n"
                f"int {name}({params_c}) {{{decl_block}{assume_block}{pre_c}\n"
                f"  {lv} = {s_c};\n  while ({lv} {direction} {e_c}) {{\n"
                f"{chr(10).join(body_lines)}\n    {lv} += {st_c};\n  }}\n  return 0;\n}}\n"
                f"{main_fn}"
            )
            result.c_status = "ok"; result.c_approximations = approx
            result.c_quality = finalize_quality(approx); return result

        raise CTranslationError("No translatable loop found")

    except CTranslationError as e:
        result.c_status="failed"; result.c_fail_reason=str(e); result.c_quality="rejected"
    except Exception as e:
        result.c_status="failed"; result.c_fail_reason=f"{type(e).__name__}: {e}"; result.c_quality="rejected"
    return result


print("C translator defined.")

C translator defined.


In [20]:
# ── PYTHON → ITS (KoAT format) TRANSLATOR ─────────────────────────────────────

def translate_to_its(impl, feat) -> TranslationResult:
    result = TranslationResult(zid=impl.zid)
    try:
        tree    = ast.parse(impl.python_code)
        fn_defs = [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
        if not fn_defs: raise CTranslationError("No function definition")
        fn = fn_defs[0]; params = [a.arg for a in fn.args.args]

        has_mod = uses_modulo(fn)
        result.its_has_precond = has_mod

        def build_ari(all_vars, after_vars, loop_guard, start_guard, start_rhs=None):
            """ARI/LCTRS format (format LCTRS :smtlib 2.6) per termCOMP 2025."""
            if start_rhs is None:
                start_rhs = list(all_vars)
            n = len(all_vars)

            def _fop(s):
                depth = 0; i = 0
                while i < len(s):
                    c = s[i]
                    if c == "(": depth += 1; i += 1
                    elif c == ")": depth -= 1; i += 1
                    elif depth > 0: i += 1
                    elif s[i:i+2] in (">=", "<=", "!="): return i, s[i:i+2]
                    elif c in (">", "<", "="): return i, c
                    elif c in ("+", "*", "/", "%"): return i, c
                    elif c == "-":
                        bef = s[:i].rstrip()
                        if bef and (bef[-1].isalnum() or bef[-1] in ("_", ")")): return i, "-"
                        else: i += 1
                    else: i += 1
                return None, None

            def _fmt(op, l, r):
                if op == "/": return f"(div {l} {r})"
                if op == "%": return f"(mod {l} {r})"
                if op == "!=": return f"(not (= {l} {r}))"
                return f"({op} {l} {r})"

            def _p(s):
                s = s.strip()
                if not s: return s
                if re.match(r"^-?\d+$", s): return s
                if re.match(r"^[A-Za-z_]\w*$", s): return s
                if s[0] == "(" and s[-1] == ")":
                    inn = s[1:-1].strip()
                    if re.match(r"^-?\d+$", inn): return inn
                    if re.match(r"^[A-Za-z_]\w*$", inn): return inn
                    p, op = _fop(inn)
                    if p is not None: return _fmt(op, _p(inn[:p]), _p(inn[p+len(op):]))
                    return _p(inn)
                p, op = _fop(s)
                if p is not None:
                    lx, rx = s[:p].strip(), s[p+len(op):].strip()
                    if lx: return _fmt(op, _p(lx), _p(rx))
                    return f"(- {_p(rx)})"
                return s

            def _split_and(s):
                parts = []; depth = 0; st = 0; i = 0
                while i < len(s):
                    if s[i] == "(": depth += 1; i += 1
                    elif s[i] == ")": depth -= 1; i += 1
                    elif depth == 0 and s[i:i+2] == "&&":
                        parts.append(s[st:i].strip()); i += 2; st = i
                    else: i += 1
                parts.append(s[st:].strip())
                return [p for p in parts if p]

            def _guard(g):
                g = g.strip()
                if g == "TRUE": return None
                conds = [_p(c) for c in _split_and(g)]
                if len(conds) == 1: return conds[0]
                return "(and " + " ".join(conds) + ")"

            def _tv(vs):
                return " ".join(f"({v} Int)" for v in vs)

            sg = _guard(start_guard)
            lg = _guard(loop_guard)
            sr = [_p(a) for a in start_rhs]
            ar = [_p(a) for a in after_vars]
            av = " ".join(all_vars)
            tv_all = _tv(all_vars)

            sr_rule = f"(rule (start {av}) (loop {' '.join(sr)})"
            if sg: sr_rule += f" :guard {sg}"
            sr_rule += ")"  # FIX: omit typed-var annotation (Int sorts inferable; it crashes its-conversion)

            lr_rule = f"(rule (loop {av}) (loop {' '.join(ar)})"
            if lg: lr_rule += f" :guard {lg}"
            lr_rule += ")"  # FIX: omit typed-var annotation (Int sorts inferable; it crashes its-conversion)

            return (
                f"(format LCTRS :smtlib 2.6)\n"
                f"(theory Ints)\n"
                f"(fun start {n})\n"
                f"(fun loop {n})\n"
                f"(entrypoint start)\n"
                f"{sr_rule}\n"
                f"{lr_rule}\n"
            )
        def write_meta(all_vars, after_vars, loop_guard, start_guard,
                       quality, approx, mod_abs):
            lines = [
                f"ZID: {impl.zid}",
                f"Label: {impl.label}",
                f"Translation quality: {quality}",
                f"Approximated statements: {approx}",
                f"Modulo abstracted: {mod_abs}",
                f"Has precondition: {has_mod}",
                f"Start guard: {start_guard}",
                f"Loop guard: {loop_guard}",
                f"Variables: {', '.join(all_vars)}",
                f"After-state: {', '.join(after_vars)}",
            ]
            if has_mod:
                lines += [
                    "Precondition: all inputs assumed >= 0",
                    "  Reason: Python % and C % differ for negative operands",
                ]
            if mod_abs:
                lines += [
                    "Modulo abstraction:",
                    "  x % y replaced by fresh variable r with r>=0 && r<y",
                    "  Sound because for x>=0, y>0: 0 <= x%y < y always holds",
                    "  Expected ranking function: second loop argument",
                ]
            (ITS_META_DIR / f"{impl.zid}.meta.txt").write_text(
                "\n".join(lines)+"\n", encoding="utf-8")

        def extract_updates(body, guard_node):
            """Extract variable updates from loop body. Raises on unsafe statements."""
            update_map = {}; modified = []; approx = 0
            loop_relevant = names_read(guard_node) | names_written(
                ast.Module(body=list(body), type_ignores=[]))
            for stmt in body:
                if (isinstance(stmt, ast.Assign) and len(stmt.targets)==1
                        and isinstance(stmt.targets[0], ast.Name)):
                    v = stmt.targets[0].id
                    update_map[v] = python_expr_to_its(stmt.value)
                    if v not in modified: modified.append(v)
                elif (isinstance(stmt, ast.Assign) and len(stmt.targets)==1
                        and isinstance(stmt.targets[0], ast.Tuple)
                        and isinstance(stmt.value, ast.Tuple)
                        and len(stmt.targets[0].elts)==len(stmt.value.elts)):
                    for lhs, rhs in zip(stmt.targets[0].elts, stmt.value.elts):
                        if not isinstance(lhs, ast.Name):
                            raise CTranslationError("ITS tuple target not a simple variable")
                        v = lhs.id
                        update_map[v] = python_expr_to_its(rhs)
                        if v not in modified: modified.append(v)
                elif (isinstance(stmt, ast.AugAssign)
                        and isinstance(stmt.target, ast.Name)):
                    v = stmt.target.id
                    op = {ast.Add:"+",ast.Sub:"-",ast.Mult:"*",
                          ast.Mod:"%",ast.FloorDiv:"/",ast.Div:"/"}.get(type(stmt.op))
                    if not op:
                        raise CTranslationError(f"Unsupported ITS aug-assign: {type(stmt.op).__name__}")
                    update_map[v] = f"({v} {op} {python_expr_to_its(stmt.value)})"
                    if v not in modified: modified.append(v)
                elif side_effect_free_ignorable(stmt) and not (names_written(stmt) & loop_relevant):
                    approx += 1
                else:
                    src = ast.unparse(stmt) if hasattr(ast, "unparse") else "<stmt>"
                    raise CTranslationError(f"unsafe_approximation in ITS: {src[:120]}")
            return update_map, modified, approx

        # ── Case 1: while loop ─────────────────────────────────────────────────
        wn = find_first_while(fn)
        if wn:
            guard_raw    = normalise_condition(python_expr_to_its(wn.test))
            body_has_mod = loop_body_has_modulo(wn.body)
            update_map, modified, approx = extract_updates(wn.body, wn.test)
            mod_abs = False

            if body_has_mod and has_mod:
                # Find the modulo divisor variable
                mod_divisor = None
                for stmt in wn.body:
                    for node in ast.walk(stmt):
                        if isinstance(node, ast.BinOp) and isinstance(node.op, ast.Mod):
                            if isinstance(node.right, ast.Name):
                                mod_divisor = node.right.id; break
                    if mod_divisor: break

                if mod_divisor:
                    # Replace the modulo result with fresh variable r
                    for v in list(update_map.keys()):
                        if "%" in update_map[v] or "//" in update_map[v]:
                            update_map[v] = "r"; break
                    all_vars = list(dict.fromkeys(params + modified))
                    if "r" not in all_vars: all_vars.append("r")
                    after_vars   = [update_map.get(v, v) for v in all_vars]
                    loop_state   = [v for v in all_vars if v != "r"]
                    start_guard, _ = precondition_for_vars(loop_state)
                    loop_guard   = f"{mod_divisor} > 0 && r >= 0 && r < {mod_divisor}"
                    approx += 1; mod_abs = True
                    quality = "sound_overapprox"
                else:
                    all_vars   = list(dict.fromkeys(params + modified))
                    after_vars = [update_map.get(v, v) for v in all_vars]
                    start_guard, _ = precondition_for_vars(all_vars)
                    loop_guard = guard_raw; quality = finalize_quality(approx)
            else:
                all_vars   = list(dict.fromkeys(params + modified))
                after_vars = [update_map.get(v, v) for v in all_vars]
                start_guard, _ = precondition_for_vars(all_vars) if has_mod else ("TRUE", "")
                loop_guard = guard_raw; quality = finalize_quality(approx)

            if not all_vars: raise CTranslationError("No variables for ITS transition")
            result.its_code             = build_ari(all_vars, after_vars, loop_guard, start_guard)
            result.its_status           = "ok"
            result.its_approximations   = approx
            result.its_quality          = quality
            result.its_modulo_abstracted = mod_abs
            write_meta(all_vars, after_vars, loop_guard, start_guard, quality, approx, mod_abs)
            return result

        # ── Case 2: for x in range(...) ────────────────────────────────────────
        for_loops = [s for s in ast.walk(fn)
                     if isinstance(s,ast.For) and isinstance(s.iter,ast.Call)
                     and isinstance(s.iter.func,ast.Name) and s.iter.func.id=="range"]
        if for_loops:
            fl = for_loops[0]
            if not isinstance(fl.target, ast.Name):
                raise CTranslationError("range target not a simple variable")
            lv = fl.target.id
            s_e, e_e, st_e, direction = range_parts(fl.iter.args, python_expr_to_its)
            synth = ast.Compare(
                left=ast.Name(id=lv, ctx=ast.Load()),
                ops=[ast.Lt() if direction=="<" else ast.Gt()],
                comparators=[ast.Name(id="_end", ctx=ast.Load())]
            )
            update_map, modified, approx = extract_updates(fl.body, synth)
            all_vars  = list(dict.fromkeys(params + [lv] + modified))
            update_map[lv] = f"({lv} + {st_e})"
            after_vars  = [update_map.get(v, v) for v in all_vars]
            loop_guard  = f"{lv} {direction} {e_e}"
            start_guard, _ = precondition_for_vars(all_vars) if has_mod else ("TRUE", "")
            start_vars  = [s_e if v==lv else v for v in all_vars]
            quality     = finalize_quality(approx)
            vl          = " ".join(all_vars)
            result.its_code = build_ari(all_vars, after_vars, loop_guard, start_guard, start_rhs=start_vars)
            result.its_status           = "ok"
            result.its_approximations   = approx
            result.its_quality          = quality
            result.its_modulo_abstracted = False
            write_meta(all_vars, after_vars, loop_guard, start_guard, quality, approx, False)
            return result

        raise CTranslationError("No while loop or for-range loop found")

    except CTranslationError as e:
        result.its_status="failed"; result.its_fail_reason=str(e); result.its_quality="rejected"
    except Exception as e:
        result.its_status="failed"; result.its_fail_reason=f"{type(e).__name__}: {e}"; result.its_quality="rejected"
    return result


print("ITS translator defined.")

ITS translator defined.


In [21]:
# ─── PYTHON → T2 FORMAT TRANSLATOR ───────────────────────────────────────
def translate_to_t2(impl, feat) -> str:
    """Generate T2 native format (.t2) for T2_static --input_t2."""
    tree    = ast.parse(impl.python_code)
    fn_defs = [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
    if not fn_defs: raise CTranslationError("No function definition")
    fn     = fn_defs[0]
    params = [a.arg for a in fn.args.args]

    def expr_t2(node):
        return python_expr_to_c(node)

    _AUG_OPS = {
        ast.Add: '+', ast.Sub: '-', ast.Mult: '*',
        ast.Div: '/', ast.FloorDiv: '/', ast.Mod: '%',
    }

    def stmt_to_t2_lines(stmt):
        out = []
        if isinstance(stmt, ast.Assign):
            if (len(stmt.targets) == 1
                    and isinstance(stmt.targets[0], ast.Name)):
                out.append(f"  {stmt.targets[0].id} := {expr_t2(stmt.value)};")
            elif (len(stmt.targets) == 1
                    and isinstance(stmt.targets[0], ast.Tuple)
                    and isinstance(stmt.value, ast.Tuple)):
                tgts = [n.id for n in stmt.targets[0].elts
                        if isinstance(n, ast.Name)]
                vals = stmt.value.elts
                temps = [f"_t{i}" for i in range(len(tgts))]
                for t, v in zip(temps, vals):
                    out.append(f"  {t} := {expr_t2(v)};")
                for tgt, tmp in zip(tgts, temps):
                    out.append(f"  {tgt} := {tmp};")
            else:
                raise CTranslationError("T2: unsupported assignment shape")
        elif isinstance(stmt, ast.AugAssign) and isinstance(stmt.target, ast.Name):
            op = _AUG_OPS.get(type(stmt.op), '?')
            out.append(f"  {stmt.target.id} := {stmt.target.id} {op} ({expr_t2(stmt.value)});")
        else:
            raise CTranslationError(f"T2: unsupported stmt {type(stmt).__name__}")
        return out

    wn = next((n for n in ast.walk(fn) if isinstance(n, ast.While)), None)
    fl = next((n for n in ast.walk(fn)
               if isinstance(n, ast.For)
               and isinstance(n.iter, ast.Call)
               and isinstance(n.iter.func, ast.Name)
               and n.iter.func.id == "range"), None)
    if not wn and not fl:
        raise CTranslationError("T2: no recognized loop")

    loop_node = wn or fl
    pre_stmts = [s for s in fn.body
                 if not isinstance(s, (ast.While, ast.For, ast.Return))
                 and s is not loop_node]

    lines = ["START: 0;", ""]

    lines.append("FROM: 0;")
    for p in params:
        lines.append(f"  {p} := nondet();")
    if uses_modulo(fn):
        for p in params:
            lines.append(f"  assume({p} >= 0);")
    lines.append("TO: 1;")
    lines.append("")

    if wn:
        cond = normalise_condition(expr_t2(wn.test))
        lines.append("FROM: 1;")
        for s in pre_stmts:
            for ln in stmt_to_t2_lines(s): lines.append(ln)
        lines.append("TO: 2;")
        lines.append("")
        lines.append("FROM: 2;")
        lines.append(f"  assume({cond});")
        for s in wn.body:
            for ln in stmt_to_t2_lines(s): lines.append(ln)
        lines.append("TO: 2;")
        lines.append("")
        lines.append("FROM: 2;")
        lines.append(f"  assume(!({cond}));")
        lines.append("TO: 3;")
    else:
        lv = fl.target.id if isinstance(fl.target, ast.Name) else "i"
        s_c, e_c, st_c, direction = range_parts(fl.iter.args, expr_t2)
        lines.append("FROM: 1;")
        for s in pre_stmts:
            for ln in stmt_to_t2_lines(s): lines.append(ln)
        lines.append(f"  {lv} := {s_c};")
        lines.append("TO: 2;")
        lines.append("")
        lines.append("FROM: 2;")
        lines.append(f"  assume({lv} {direction} {e_c});")
        for s in fl.body:
            for ln in stmt_to_t2_lines(s): lines.append(ln)
        lines.append(f"  {lv} := {lv} + ({st_c});")
        lines.append("TO: 2;")
        lines.append("")
        lines.append("FROM: 2;")
        lines.append(f"  assume(!({lv} {direction} {e_c}));")
        lines.append("TO: 3;")

    return "\n".join(lines) + "\n"

print("T2 translator defined.")


T2 translator defined.


In [22]:
# ── SANITY CHECK: GCD ──────────────────────────────────────────────────────────
# Verify both translators work correctly on the thesis running example.
#
# Expected C:
#   int x=0; int y=0; declared, __VERIFIER_assume for both params,
#   while(y != 0), temporaries _t0/_t1, quality=exact, approx=0
#
# Expected ITS:
#   (GOAL COMPLEXITY) as first line — no semicolons
#   loop guard: y > 0 && r >= 0 && r < y
#   quality=sound_overapprox, modulo_abstracted=True
#
# Expected .meta.txt:
#   Written to its_meta/Z13642.meta.txt

GCD_PY = """def Z13612(Z13612K1, Z13612K2):
    if (Z13612K1==0): return Z13612K2
    x = Z13612K1
    y = Z13612K2
    while(y):
        x, y = y, x % y
    return x"""

gcd_impl = WFImplementation(zid="Z13642", function_zid="Z13612",
                             python_code=GCD_PY, label="GCD, python")
gcd_feat = FunctionFeatures(zid="Z13642", has_while=True,
                             has_int_arithmetic=True, analysis_category="INT_LOOP_CLEAN")

gcd_c   = translate_to_c(gcd_impl, gcd_feat)
gcd_its = translate_to_its(gcd_impl, gcd_feat)

print("── GCD C ─────────────────────────────────────────────")
print(gcd_c.c_code)
print("── GCD ITS (.koat — must start with (GOAL COMPLEXITY)) ")
print(gcd_its.its_code)
print("── GCD metadata (.meta.txt) ──────────────────────────")
meta_f = ITS_META_DIR / "Z13642.meta.txt"
if meta_f.exists(): print(meta_f.read_text())
print(f"C:   status={gcd_c.c_status}  quality={gcd_c.c_quality}  approx={gcd_c.c_approximations}")
print(f"ITS: status={gcd_its.its_status}  quality={gcd_its.its_quality}  mod_abs={gcd_its.its_modulo_abstracted}")

ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ GCD C ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡

In [23]:
if REBUILD_FROM_SOURCE:
    # ── RUN TRANSLATIONS ───────────────────────────────────────────────────────────
    TRANSLATE_CATS = {"INT_LOOP_TRANSLATABLE"}  # faithful set only (C, ITS, T2, KoAT)
    translations: dict = {}
    feat_by_zid = {f.zid: f for f in features_list}

    # Clear output dirs so each run starts clean (prevents stale files from earlier runs
    # from being fed to the tools). KoAT's .koat dir is cleared in the CFG cell below.
    for _d in [C_OUT_DIR, ITS_OUT_DIR, T2_OUT_DIR, C_INVALID_DIR]:
        _d.mkdir(parents=True, exist_ok=True)
        for _old in _d.glob("*"):
            if _old.is_file():
                _old.unlink()

    for impl in tqdm(implementations, desc="Translating"):
        feat = feat_by_zid[impl.zid]
        cat  = feat.analysis_category
        if cat not in TRANSLATE_CATS:
            tr = TranslationResult(
                zid=impl.zid, c_status="skipped", its_status="skipped",
                c_quality="not_attempted", its_quality="not_attempted",
                c_fail_reason=f"Category: {cat}", its_fail_reason=f"Category: {cat}",
            )
        else:
            tr_c   = translate_to_c(impl, feat)
            tr_its = translate_to_its(impl, feat)
            try:
                t2_code   = translate_to_t2(impl, feat)
                t2_status = "ok"
                t2_fail   = ""
            except Exception as _t2e:
                t2_code   = None
                t2_status = "failed"
                t2_fail   = str(_t2e)
            tr = TranslationResult(
                zid=impl.zid,
                c_code=tr_c.c_code, c_status=tr_c.c_status,
                c_fail_reason=tr_c.c_fail_reason, c_approximations=tr_c.c_approximations,
                c_quality=tr_c.c_quality, c_has_precond=tr_c.c_has_precond,
                its_code=tr_its.its_code, its_status=tr_its.its_status,
                its_fail_reason=tr_its.its_fail_reason, its_approximations=tr_its.its_approximations,
                its_quality=tr_its.its_quality, its_has_precond=tr_its.its_has_precond,
                its_modulo_abstracted=tr_its.its_modulo_abstracted,
                t2_code=t2_code, t2_status=t2_status, t2_fail_reason=t2_fail,
            )
            if tr.c_code:   (C_OUT_DIR   / f"{impl.zid}.c").write_text(tr.c_code,   encoding="utf-8")
            if tr.its_code: (ITS_OUT_DIR / f"{impl.zid}.ari").write_text(tr.its_code,  encoding="utf-8")
            if tr.t2_code:  (T2_OUT_DIR  / f"{impl.zid}.t2").write_text(tr.t2_code,   encoding="utf-8")
        translations[impl.zid] = tr

    print("\n── Translation summary ────────────────────────────────")
    for fmt, sq, iq in [("C","c_status","c_quality"),("ITS","its_status","its_quality")]:
        ok      = sum(1 for t in translations.values() if getattr(t,sq)=="ok")
        exact   = sum(1 for t in translations.values() if getattr(t,iq)=="exact")
        sa      = sum(1 for t in translations.values() if getattr(t,iq)=="sound_overapprox")
        rej     = sum(1 for t in translations.values() if getattr(t,iq)=="rejected")
        print(f"  {fmt}: ok={ok}  exact={exact}  sound_overapprox={sa}  rejected={rej}")
    print(f"  ITS modulo-abstracted: {sum(1 for t in translations.values() if t.its_modulo_abstracted)}")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping translation pass; translations already committed in the DB.")


REBUILD_FROM_SOURCE is False -- skipping translation pass; translations already committed in the DB.


In [24]:
# ============================================================
# CFG -> {ITS(ARI), C, T2, KoAT} translator for INT_LOOP_TRANSLATABLE
# and INT_LOOP_COMPLEX functions. Conditionals -> guarded transitions;
# multiple/nested loops -> multiple locations. Each function is kept ONLY
# if a differential test (Python vs ITS interpreter on random inputs) matches.
# ============================================================
import ast, sqlite3, random, signal

# ----------------------------------------------------------------------------
# CFG -> ITS translator: locations + guarded transitions.
# Each transition: (src, guard_ast_or_None, updates{var:ast}, dst)
# Straight-line assigns chained via fresh locations (sequential semantics).
# Tuple assign -> one simultaneous-update transition.
# if/while/for-range/return/break/continue handled structurally.
# ----------------------------------------------------------------------------

class Unsupported(Exception): pass

class CFG:
    def __init__(self):
        self.t=[]; self.n=0
        self.HALT=self.new()
    def new(self):
        self.n+=1; return self.n
    def add(self,src,guard,updates,dst):
        self.t.append((src,guard,updates,dst))

def _notexpr(e):
    return ast.UnaryOp(op=ast.Not(), operand=e)

def build_cfg(fn):
    cfg=CFG()
    start=cfg.new()
    # compile body; normal completion (fall off end) -> HALT (implicit return None)
    def comp(stmts, entry, after, loopctx):
        cur=entry
        for s in stmts:
            nxt=cfg.new()
            cur2=comp_stmt(s, cur, nxt, loopctx)
            cur=nxt
        # link last to after
        cfg.add(cur, None, {}, after)
        return entry
    def comp_stmt(s, entry, after, loopctx):
        if isinstance(s, ast.Assign):
            if len(s.targets)==1 and isinstance(s.targets[0], ast.Name):
                cfg.add(entry, None, {s.targets[0].id: s.value}, after)
            elif (len(s.targets)==1 and isinstance(s.targets[0], ast.Tuple)
                  and isinstance(s.value, ast.Tuple)
                  and len(s.targets[0].elts)==len(s.value.elts)):
                upd={}
                for lhs,rhs in zip(s.targets[0].elts, s.value.elts):
                    if not isinstance(lhs, ast.Name): raise Unsupported("tuple target")
                    upd[lhs.id]=rhs
                cfg.add(entry, None, upd, after)
            else: raise Unsupported("assign shape")
        elif isinstance(s, ast.AugAssign) and isinstance(s.target, ast.Name):
            op=s.op
            newval=ast.BinOp(left=ast.Name(id=s.target.id,ctx=ast.Load()), op=op, right=s.value)
            cfg.add(entry, None, {s.target.id: newval}, after)
        elif isinstance(s, ast.If):
            tb=cfg.new(); comp(s.body, tb, after, loopctx)
            if s.orelse:
                fb=cfg.new(); comp(s.orelse, fb, after, loopctx)
            else:
                fb=after
            cfg.add(entry, s.test, {}, tb)
            cfg.add(entry, _notexpr(s.test), {}, fb)
        elif isinstance(s, ast.While):
            head=cfg.new()
            cfg.add(entry, None, {}, head)
            body=cfg.new(); comp(s.body, body, head, (head, after))
            cfg.add(head, s.test, {}, body)
            cfg.add(head, _notexpr(s.test), {}, after)
        elif isinstance(s, ast.For):
            # for v in range(a,b,step)  -> v=a; while v<b (or >): body; v+=step
            if not (isinstance(s.iter, ast.Call) and isinstance(s.iter.func, ast.Name)
                    and s.iter.func.id=="range" and isinstance(s.target, ast.Name)):
                raise Unsupported("for non-range")
            v=s.target.id; args=s.iter.args
            if len(args)==1: a=ast.Constant(0); b=args[0]; step=ast.Constant(1)
            elif len(args)==2: a,b=args; step=ast.Constant(1)
            elif len(args)==3: a,b,step=args
            else: raise Unsupported("range args")
            # determine direction from step if constant
            def _cint(nd):
                if isinstance(nd,ast.Constant) and isinstance(nd.value,int): return nd.value
                if isinstance(nd,ast.UnaryOp) and isinstance(nd.op,ast.USub) and isinstance(nd.operand,ast.Constant): return -nd.operand.value
                return None
            _sv=_cint(step); neg = _sv is not None and _sv<0
            cmp = ast.Gt() if neg else ast.Lt()
            init=cfg.new(); cfg.add(entry, None, {v:a}, init)
            head=cfg.new(); cfg.add(init, None, {}, head)
            test=ast.Compare(left=ast.Name(id=v,ctx=ast.Load()), ops=[cmp], comparators=[b])
            body=cfg.new()
            inc=cfg.new()
            comp(s.body, body, inc, (head, after))
            cfg.add(inc, None, {v: ast.BinOp(left=ast.Name(id=v,ctx=ast.Load()), op=ast.Add(), right=step)}, head)
            cfg.add(head, test, {}, body)
            cfg.add(head, _notexpr(test), {}, after)
        elif isinstance(s, ast.Return):
            val=s.value if s.value is not None else ast.Constant(0)
            cfg.add(entry, None, {"__ret": val, "__done": ast.Constant(1)}, cfg.HALT)
        elif isinstance(s, ast.Break):
            cfg.add(entry, None, {}, loopctx[1])
        elif isinstance(s, ast.Continue):
            cfg.add(entry, None, {}, loopctx[0])
        elif isinstance(s, ast.Expr):  # docstring / bare expr - no-op
            cfg.add(entry, None, {}, after)
        elif isinstance(s, (ast.Import, ast.ImportFrom, ast.Pass)):
            cfg.add(entry, None, {}, after)
        else:
            raise Unsupported(f"stmt {type(s).__name__}")
        return after
    comp(fn.body, start, cfg.HALT, None)
    return cfg, start

# ---------------- expression evaluation (interpreter) ----------------
def ev(node, env):
    if isinstance(node, ast.Constant): return node.value
    if isinstance(node, ast.Name): return env.get(node.id, 0)
    if isinstance(node, ast.BinOp):
        a=ev(node.left,env); b=ev(node.right,env); op=node.op
        if isinstance(op,ast.Add): return a+b
        if isinstance(op,ast.Sub): return a-b
        if isinstance(op,ast.Mult): return a*b
        if isinstance(op,ast.FloorDiv):
            if b==0: raise ZeroDivisionError()
            return a//b
        if isinstance(op,ast.Mod):
            if b==0: raise ZeroDivisionError()
            return a%b
        if isinstance(op,ast.Pow): return a**b
        if isinstance(op,ast.LShift): return a<<b
        if isinstance(op,ast.RShift): return a>>b
        raise Unsupported("binop ev")
    if isinstance(node, ast.UnaryOp):
        if isinstance(node.op,ast.Not): return not ev(node.operand,env)
        if isinstance(node.op,ast.USub): return -ev(node.operand,env)
        if isinstance(node.op,ast.UAdd): return +ev(node.operand,env)
        raise Unsupported("unaryop ev")
    if isinstance(node, ast.BoolOp):
        vals=[ev(v,env) for v in node.values]
        if isinstance(node.op,ast.And): return all(vals)
        return any(vals)
    if isinstance(node, ast.Compare):
        left=ev(node.left,env); res=True
        for op,comp in zip(node.ops,node.comparators):
            r=ev(comp,env)
            if isinstance(op,ast.Lt): res=res and (left<r)
            elif isinstance(op,ast.LtE): res=res and (left<=r)
            elif isinstance(op,ast.Gt): res=res and (left>r)
            elif isinstance(op,ast.GtE): res=res and (left>=r)
            elif isinstance(op,ast.Eq): res=res and (left==r)
            elif isinstance(op,ast.NotEq): res=res and (left!=r)
            else: raise Unsupported("cmp ev")
            left=r
        return res
    raise Unsupported(f"expr ev {type(node).__name__}")

def its_run(cfg, start, params, inputs, allvars, max_steps=40000):
    env={v:0 for v in allvars}
    for p,val in zip(params, inputs): env[p]=val
    # index transitions by src
    from collections import defaultdict
    bysrc=defaultdict(list)
    for (s,g,u,d) in cfg.t: bysrc[s].append((g,u,d))
    loc=start; steps=0
    while loc!=cfg.HALT and steps<max_steps:
        outs=bysrc[loc]
        fired=None
        for (g,u,d) in outs:
            try: ok = True if g is None else bool(ev(g,env))
            except ZeroDivisionError: ok=False
            if ok: fired=(g,u,d); break
        if fired is None:
            # no enabled transition: stuck = halt (treat as termination)
            return True, steps, env.get("__ret",None)
        g,u,d=fired
        try:
            newvals={var: ev(expr,env) for var,expr in u.items()}
        except ZeroDivisionError:
            return True, steps, env.get("__ret",None)  # would error in python too
        if any(isinstance(v,int) and abs(v)>10**12 for v in newvals.values()):
            return None, steps, None   # numeric blow-up -> skip this input
        env.update(newvals); loc=d; steps+=1
    if loc==cfg.HALT: return True, steps, env.get("__ret",None)
    return False, steps, None   # step budget exceeded -> treat as nonterminating

# ---------------- python runner: portable line-count budget (settrace) ----------------
import sys as _sys
class StepExceeded(Exception): pass
def py_run(code, fname, inputs, budget=40000):
    ns={}
    try: exec(code, ns)
    except Exception: return ("err","compile")
    fn=ns[fname]; cnt=[0]
    def lt(frame,event,arg):
        if event=='line':
            cnt[0]+=1
            if cnt[0]>budget: raise StepExceeded()
        return lt
    def gt(frame,event,arg): return lt
    old=_sys.gettrace(); _sys.settrace(gt)
    try:
        r=fn(*inputs); return ("ret",r)
    except StepExceeded: return ("timeout",None)
    except ZeroDivisionError: return ("zerodiv",None)
    except RecursionError: return ("timeout",None)
    except Exception as e: return ("err",type(e).__name__)
    finally: _sys.settrace(old)

def collect_vars(fn):
    vs=[a.arg for a in fn.args.args]
    for n in ast.walk(fn):
        if isinstance(n,ast.Name) and isinstance(n.ctx,ast.Store): 
            if n.id not in vs: vs.append(n.id)
        if isinstance(n,ast.AugAssign) and isinstance(n.target,ast.Name) and n.target.id not in vs:
            vs.append(n.target.id)
    for extra in ("__ret","__done"):
        if extra not in vs: vs.append(extra)
    return vs

def difftest(code, ntrials=10):
    t=ast.parse(code); fn=[n for n in ast.walk(t) if isinstance(n,ast.FunctionDef)][0]
    params=[a.arg for a in fn.args.args]
    cfg,start=build_cfg(fn)
    allvars=collect_vars(fn)
    mism=0; checked=0; both_term=0
    random.seed(1234)
    inputsets=[]
    for _ in range(ntrials):
        inputsets.append([random.choice([random.randint(0,7), random.randint(0,3), random.randint(0,9)]) for _ in params])
    # add small structured inputs
    for base in (0,1,2,3,4,5,6,7,8,-1,-2):
        inputsets.append([base for _ in params])
    for inp in inputsets:
        pr=py_run(code, fn.name, inp)
        if pr[0] in ("err",):  # python itself errored (e.g. on weird input) - skip
            continue
        term, steps, ret = its_run(cfg,start,params,inp,allvars)
        if term is None: continue  # ITS numeric blow-up; skip
        checked+=1
        if pr[0]=="ret":
            both_term+=1
            if not term: mism+=1; 
            elif ret is not None and ret!=pr[1] and isinstance(pr[1],(int,bool)):
                # value mismatch (only compare when python returned int/bool)
                mism+=1
        elif pr[0]=="timeout":
            if term: mism+=1   # python loops, ITS halted -> mismatch
        elif pr[0]=="zerodiv":
            pass  # both should error; we treat as halt; skip strict check
    return checked, both_term, mism, len(cfg.t), cfg.n

# ==================== ARI (multi-location ITS) emitter ====================
class EmitError(Exception): pass

def _pow_expand(base_s, k):
    if k==0: return "1"
    return "(* " + " ".join([base_s]*k) + ")" if k>1 else base_s

def _as_bool(node):
    if isinstance(node,(ast.Compare,ast.BoolOp)): return node
    if isinstance(node,ast.UnaryOp) and isinstance(node.op,ast.Not): return node
    return ast.Compare(left=node, ops=[ast.NotEq()], comparators=[ast.Constant(0)])
def to_sexpr(node, in_guard):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, bool): return "true" if node.value else "false"
        if isinstance(node.value, int): return str(node.value)
        raise EmitError("non-int const")
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.UnaryOp):
        if isinstance(node.op, ast.Not): return f"(not {to_sexpr(node.operand,True)})"
        if isinstance(node.op, ast.USub): return f"(- {to_sexpr(node.operand,in_guard)})"
        if isinstance(node.op, ast.UAdd): return to_sexpr(node.operand,in_guard)
        raise EmitError("unaryop")
    if isinstance(node, ast.BoolOp):
        if not in_guard: raise EmitError("boolop in update")
        op = "and" if isinstance(node.op, ast.And) else "or"
        return f"({op} " + " ".join(to_sexpr(v,True) for v in node.values) + ")"
    if isinstance(node, ast.Compare):
        if not in_guard: raise EmitError("compare in update")
        if len(node.ops)>1:  # chained -> conjunction
            parts=[]; left=node.left
            for op,comp in zip(node.ops,node.comparators):
                parts.append(_cmp(op,left,comp)); left=comp
            return "(and "+" ".join(parts)+")"
        return _cmp(node.ops[0], node.left, node.comparators[0])
    if isinstance(node, ast.BinOp):
        l=node.left; r=node.right; op=node.op
        if isinstance(op, ast.Pow):
            if not (isinstance(r,ast.Constant) and isinstance(r.value,int) and r.value>=0): raise EmitError("pow")
            return _pow_expand(to_sexpr(l,in_guard), r.value)
        if isinstance(op, ast.LShift):
            if not (isinstance(r,ast.Constant) and isinstance(r.value,int)): raise EmitError("shift")
            return f"(* {to_sexpr(l,in_guard)} {2**r.value})"
        if isinstance(op, ast.RShift):
            if not (isinstance(r,ast.Constant) and isinstance(r.value,int)): raise EmitError("shift")
            return f"(div {to_sexpr(l,in_guard)} {2**r.value})"
        m={ast.Add:"+",ast.Sub:"-",ast.Mult:"*",ast.FloorDiv:"div",ast.Mod:"mod"}
        if type(op) in m: return f"({m[type(op)]} {to_sexpr(l,in_guard)} {to_sexpr(r,in_guard)})"
        raise EmitError("binop "+type(op).__name__)
    raise EmitError("expr "+type(node).__name__)

def _cmp(op, l, r):
    ls=to_sexpr(l,True); rs=to_sexpr(r,True)
    if isinstance(op,ast.Lt): return f"(< {ls} {rs})"
    if isinstance(op,ast.LtE): return f"(<= {ls} {rs})"
    if isinstance(op,ast.Gt): return f"(> {ls} {rs})"
    if isinstance(op,ast.GtE): return f"(>= {ls} {rs})"
    if isinstance(op,ast.Eq): return f"(= {ls} {rs})"
    if isinstance(op,ast.NotEq): return f"(not (= {ls} {rs}))"
    raise EmitError("cmp")

def emit_ari(fn):
    cfg,start=build_cfg(fn)
    allv=[v for v in collect_vars(fn) if v not in ("__ret","__done")]
    # locations actually reachable / used
    locs=set([start, cfg.HALT])
    for (s,g,u,d) in cfg.t: locs.add(s); locs.add(d)
    n=len(allv)
    lines=["(format LCTRS :smtlib 2.6)","(theory Ints)"]
    for L in sorted(locs): lines.append(f"(fun loc{L} {n})")
    lines.append(f"(entrypoint loc{start})")
    av=" ".join(allv)
    for (s,g,u,d) in cfg.t:
        if s==cfg.HALT: continue
        rhs=[]
        for v in allv:
            if v in u: rhs.append(to_sexpr(u[v], False))
            else: rhs.append(v)
        rule=f"(rule (loc{s} {av}) (loc{d} {' '.join(rhs)})"
        if g is not None:
            rule += f" :guard {to_sexpr(_as_bool(g), True)}"
        rule += ")"
        lines.append(rule)
    return "\n".join(lines)+"\n"

# ==================== C emitter (goto-CFG) ====================
def to_c(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value,bool): return "1" if node.value else "0"
        if isinstance(node.value,int): return str(node.value)
        raise EmitError("c const")
    if isinstance(node, ast.Name): return node.id
    if isinstance(node, ast.UnaryOp):
        if isinstance(node.op,ast.Not): return f"(!({to_c(node.operand)}))"
        if isinstance(node.op,ast.USub): return f"(-({to_c(node.operand)}))"
        if isinstance(node.op,ast.UAdd): return to_c(node.operand)
        raise EmitError("c unary")
    if isinstance(node, ast.BoolOp):
        op="&&" if isinstance(node.op,ast.And) else "||"
        return "("+f" {op} ".join(to_c(v) for v in node.values)+")"
    if isinstance(node, ast.Compare):
        if len(node.ops)>1:
            parts=[]; left=node.left
            for op,comp in zip(node.ops,node.comparators):
                parts.append(_cmpc(op,left,comp)); left=comp
            return "("+" && ".join(parts)+")"
        return _cmpc(node.ops[0],node.left,node.comparators[0])
    if isinstance(node, ast.BinOp):
        l,r,op=node.left,node.right,node.op
        if isinstance(op,ast.Pow):
            if not(isinstance(r,ast.Constant) and r.value>=0): raise EmitError("c pow")
            return "("+("*".join([f"({to_c(l)})"]*r.value) if r.value>0 else "1")+")"
        if isinstance(op,ast.LShift): return f"(({to_c(l)})*{2**r.value})"
        if isinstance(op,ast.RShift): return f"(({to_c(l)})/{2**r.value})"
        m={ast.Add:"+",ast.Sub:"-",ast.Mult:"*",ast.FloorDiv:"/",ast.Mod:"%"}
        return f"(({to_c(l)}) {m[type(op)]} ({to_c(r)}))"
    raise EmitError("c expr "+type(node).__name__)
def _cmpc(op,l,r):
    m={ast.Lt:"<",ast.LtE:"<=",ast.Gt:">",ast.GtE:">=",ast.Eq:"==",ast.NotEq:"!="}
    return f"({to_c(l)} {m[type(op)]} {to_c(r)})"

def emit_c(fn):
    cfg,start=build_cfg(fn)
    params=[a.arg for a in fn.args.args]
    allv=[v for v in collect_vars(fn) if v not in ("__ret","__done")]
    from collections import defaultdict
    bysrc=defaultdict(list)
    for (s,g,u,d) in cfg.t: bysrc[s].append((g,u,d))
    L=[]
    L.append("extern int __VERIFIER_nondet_int(void);")
    L.append("extern void __VERIFIER_assume(int);")
    L.append(f"int main() {{")
    for p in params: L.append(f"  long {p} = __VERIFIER_nondet_int();")
    L.append("  __VERIFIER_assume(" + " && ".join(f"{p} >= 0" for p in params) + ");" if params else "")
    for v in allv:
        if v not in params: L.append(f"  long {v} = 0;")
    def block_updates(u):
        u={k:v for k,v in u.items() if k in allv}
        if not u: return []
        if len(u)==1:
            v,e=next(iter(u.items())); return [f"    {v} = {to_c(e)};"]
        tmps=[]; out=[]
        for i,(v,e) in enumerate(u.items()):
            out.append(f"    long _t{i} = {to_c(e)};")
        for i,(v,e) in enumerate(u.items()):
            out.append(f"    {v} = _t{i};")
        return out
    locs=sorted(set([start,cfg.HALT])|{s for s in bysrc}|{d for (_,_,_,d) in cfg.t})
    L.append(f"  goto Lloc{start};")
    for loc in locs:
        L.append(f"Lloc{loc}: ;")
        if loc==cfg.HALT:
            L.append("  return 0;"); continue
        outs=bysrc.get(loc,[])
        if not outs:
            L.append("  return 0;"); continue
        if len(outs)==1 and outs[0][0] is None:
            g,u,d=outs[0]
            L+=block_updates(u); L.append(f"  goto Lloc{d};")
        else:
            for (g,u,d) in outs:
                cond = to_c(g) if g is not None else "1"
                L.append(f"  if ({cond}) {{")
                for ln in block_updates(u): L.append("  "+ln)
                L.append(f"    goto Lloc{d};")
                L.append("  }")
            L.append("  return 0;")
    L.append("}")
    return "\n".join(x for x in L if x!="")+"\n"

# ==================== T2 emitter (FROM/TO blocks) ====================
def to_t2(node): return to_c(node)  # T2 expression syntax ~ C for arithmetic/compares
def emit_t2(fn):
    cfg,start=build_cfg(fn)
    allv=[v for v in collect_vars(fn) if v not in ("__ret","__done")]
    L=["START: 0;",""]
    # map locations to ints; use loc numbers directly; entry edge 0 -> start
    L.append("FROM: 0;")
    L.append(f"TO: {start};")
    L.append("")
    for (s,g,u,d) in cfg.t:
        if s==cfg.HALT: continue
        L.append(f"FROM: {s};")
        if g is not None:
            L.append(f"assume({_t2guard(g)});")
        u={k:v for k,v in u.items() if k in allv}
        if u:
            if len(u)>1:
                for i,(v,e) in enumerate(u.items()): L.append(f"_t{i} := {to_t2(e)};")
                for i,(v,e) in enumerate(u.items()): L.append(f"{v} := _t{i};")
            else:
                v,e=next(iter(u.items())); L.append(f"{v} := {to_t2(e)};")
        L.append(f"TO: {d};")
        L.append("")
    return "\n".join(L)+"\n"
def _t2guard(node): return to_c(node)

# ==================== KoAT (native .koat) emitter ====================
_NEGOP={ast.Lt:ast.GtE, ast.LtE:ast.Gt, ast.Gt:ast.LtE, ast.GtE:ast.Lt, ast.Eq:ast.NotEq, ast.NotEq:ast.Eq}
_CMPSYM={ast.Lt:"<",ast.LtE:"<=",ast.Gt:">",ast.GtE:">=",ast.Eq:"==",ast.NotEq:"!="}
def _norm_chained(node):
    if isinstance(node, ast.Compare) and len(node.ops)>1:
        parts=[]; left=node.left
        for op,comp in zip(node.ops,node.comparators):
            parts.append(ast.Compare(left=left, ops=[op], comparators=[comp])); left=comp
        return ast.BoolOp(op=ast.And(), values=parts)
    return node
def _neg_cmp(node):
    return ast.Compare(left=node.left, ops=[_NEGOP[type(node.ops[0])]()], comparators=node.comparators)
def _dnf(node, neg):
    node=_norm_chained(node)
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.Not):
        return _dnf(node.operand, not neg)
    if isinstance(node, ast.BoolOp):
        subs=[_dnf(v,neg) for v in node.values]
        is_and=isinstance(node.op,ast.And)
        do_and=(is_and and not neg) or ((not is_and) and neg)
        if do_and:
            from itertools import product
            return [sum(map(list,combo),[]) for combo in product(*subs)]
        return [c for s in subs for c in s]
    if isinstance(node, ast.Compare):
        node=_norm_chained(node)
        if isinstance(node, ast.BoolOp): return _dnf(node, neg)
        return [[_neg_cmp(node) if neg else node]]
    # bare truthiness: treat `node` as (node != 0)
    _c = ast.Compare(left=node, ops=[ast.Eq() if neg else ast.NotEq()], comparators=[ast.Constant(0)])
    return [[_c]]
def _koat_cmp(node):
    return f"{to_c(node.left)} {_CMPSYM[type(node.ops[0])]} {to_c(node.comparators[0])}"
def emit_koat(fn):
    cfg,start=build_cfg(fn)
    allv=[v for v in collect_vars(fn) if v not in ("__ret","__done")]
    L=["(GOAL COMPLEXITY)", f"(STARTTERM (FUNCTIONSYMBOLS loc{start}))",
       f"(VAR {' '.join(allv)})", "(RULES"]
    head_args=", ".join(allv)
    for (s,g,u,d) in cfg.t:
        if s==cfg.HALT: continue
        rhs=", ".join(to_c(u[v]) if v in u else v for v in allv)
        head=f"loc{s}({head_args})"; tail=f"loc{d}({rhs})"
        if g is None:
            L.append(f"  {head} -> {tail}")
        else:
            for conj in _dnf(g, False):
                cond=" && ".join(_koat_cmp(c) for c in conj)
                L.append(f"  {head} -> {tail} :|: {cond}")
    L.append(")")
    return "\n".join(L)+"\n"


In [25]:
if REBUILD_FROM_SOURCE:
    # ── Translate faithful integer functions via the CFG translator (all 4 formats) ──
    # Runs on INT_LOOP_TRANSLATABLE and INT_LOOP_COMPLEX, gated by a differential test.
    # Emits .ari, .c, .t2 AND native .koat (KoAT is fed .koat directly -- no converter).
    import ast as _ast2
    KOAT_DIR = OUTPUT_DIR / "its_translations"
    KOAT_DIR.mkdir(parents=True, exist_ok=True)
    for _f in KOAT_DIR.glob("*.koat"):
        _f.unlink()                       # clear stale .koat
    cfg_full = 0
    for _impl in implementations:
        _feat = feat_by_zid[_impl.zid]
        if _feat.analysis_category not in ("INT_LOOP_TRANSLATABLE", "INT_LOOP_COMPLEX"):
            continue
        try:
            _ch, _bt, _mism, _ntr, _nloc = difftest(_impl.python_code)
            if _mism != 0 or _ch < 8:
                continue
            _fn = [n for n in _ast2.walk(_ast2.parse(_impl.python_code))
                   if isinstance(n, _ast2.FunctionDef)][0]
            _ari = emit_ari(_fn); _cc = emit_c(_fn); _t2 = emit_t2(_fn); _koat = emit_koat(_fn)
        except Exception:
            continue
        (ITS_OUT_DIR / f"{_impl.zid}.ari").write_text(_ari, encoding="utf-8")
        (C_OUT_DIR   / f"{_impl.zid}.c").write_text(_cc,  encoding="utf-8")
        (T2_OUT_DIR  / f"{_impl.zid}.t2").write_text(_t2,  encoding="utf-8")
        (KOAT_DIR    / f"{_impl.zid}.koat").write_text(_koat, encoding="utf-8")
        _tr = translations.get(_impl.zid)
        if _tr is None:
            _tr = TranslationResult(zid=_impl.zid); translations[_impl.zid] = _tr
        _tr.its_code=_ari; _tr.its_status="ok"; _tr.its_quality="exact"; _tr.its_fail_reason=""
        _tr.c_code=_cc;   _tr.c_status="ok";   _tr.c_quality="exact";   _tr.c_fail_reason=""
        _tr.t2_code=_t2;  _tr.t2_status="ok";  _tr.t2_fail_reason=""
        _feat.analysis_category = "INT_LOOP_TRANSLATABLE_CFG"
        cfg_full += 1
    print(f"CFG-translated to .ari/.c/.t2/.koat (differential-tested): {cfg_full}")
    print(f"Total faithful (have .ari): {sum(1 for t in translations.values() if getattr(t,'its_status','')=='ok')}")
    print(f".koat files written: {len(list(KOAT_DIR.glob('*.koat')))}")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping CFG/.koat translation pass (needs in-memory implementations/translations).")


REBUILD_FROM_SOURCE is False -- skipping CFG/.koat translation pass (needs in-memory implementations/translations).


In [26]:
if REBUILD_FROM_SOURCE:
    # Rejection reason breakdown
    c_r   = Counter(t.c_fail_reason   for t in translations.values() if t.c_status  =="failed")
    its_r = Counter(t.its_fail_reason for t in translations.values() if t.its_status=="failed")
    print("── C rejection reasons ───────────────────────────────")
    for r,n in c_r.most_common(10): print(f"  {n:>4}x  {r}")
    print("\n── ITS rejection reasons ────────────────────────────")
    for r,n in its_r.most_common(10): print(f"  {n:>4}x  {r}")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping rejection-reason breakdown (needs in-memory translations).")


REBUILD_FROM_SOURCE is False -- skipping rejection-reason breakdown (needs in-memory translations).


In [27]:
# ── Validation: ARI output sanity (informational) ──────────────────────────
from pathlib import Path as _P
_GCD_PY = """def Z13612(Z13612K1, Z13612K2):
    if (Z13612K1==0): return Z13612K2
    x = Z13612K1
    y = Z13612K2
    while(y):
        x, y = y, x % y
    return x"""
_gcd_impl = WFImplementation(zid="Z13642", function_zid="Z13612",
                             python_code=_GCD_PY, label="GCD, python")
_gcd_feat = FunctionFeatures(zid="Z13642", has_while=True,
                             has_int_arithmetic=True, analysis_category="INT_LOOP_TRANSLATABLE")
_gcd_its  = translate_to_its(_gcd_impl, _gcd_feat)
_ari_out  = _gcd_its.its_code or ""
print("── ARI output for GCD (Z13642) ─────────────────────────────────────")
print(_ari_out)

_fmt_ok  = _ari_out.startswith("(format LCTRS :smtlib 2.6)")
_novar   = ":var" not in _ari_out          # we deliberately omit :var (converter-safe)
_hasrule = "(rule " in _ari_out
print(f":smtlib 2.6 header:          {'PASS' if _fmt_ok else 'FAIL'}")
print(f"no :var (converter-safe):   {'PASS' if _novar else 'FAIL'}")
print(f"has rules:                  {'PASS' if _hasrule else 'FAIL'}")

_c_count   = len(list(_P("thesis_output/c_translations").glob("*.c")))
_ari_count = len(list(_P("thesis_output/its_ari").glob("*.ari"))) if _P("thesis_output/its_ari").exists() else 0
print(f"\n.c files in c_translations/:   {_c_count}")
print(f".ari files in its_ari/:         {_ari_count}")

assert _fmt_ok and _novar and _hasrule, "GCD ARI sanity failed"
print("\n✓  ARI sanity checks passed")


── ARI output for GCD (Z13642) ─────────────────────────────────────
(format LCTRS :smtlib 2.6)
(theory Ints)
(fun start 5)
(fun loop 5)
(entrypoint start)
(rule (start Z13612K1 Z13612K2 x y r) (loop Z13612K1 Z13612K2 x y r) :guard (and (>= Z13612K1 0) (>= Z13612K2 0) (>= x 0) (>= y 0)))
(rule (loop Z13612K1 Z13612K2 x y r) (loop Z13612K1 Z13612K2 y r r) :guard (and (> y 0) (>= r 0) (< r y)))

:smtlib 2.6 header:          PASS
no :var (converter-safe):   PASS
has rules:                  PASS

.c files in c_translations/:   43
.ari files in its_ari/:         43

✓  ARI sanity checks passed


---
## Stage 3b — gcc Validation

Every generated `.c` file is compiled with `gcc -fsyntax-only`.  
Files that fail move to `c_invalid/` and never reach AProVE or T2.

In [28]:
def validate_c_file(filepath):
    try:
        r = subprocess.run(["gcc","-fsyntax-only","-w",str(filepath)],
                           capture_output=True, text=True, timeout=10)
        if r.returncode == 0: return True, ""
        return False, (r.stderr or r.stdout).strip()[:400]
    except FileNotFoundError: return None, "gcc not found"
    except subprocess.TimeoutExpired: return False, "gcc timeout"
    except Exception as e: return False, str(e)

import shutil as _sh3
c_files = sorted(C_OUT_DIR.glob("*.c"))
valid_c, invalid_c = [], []

if _sh3.which("gcc") is None:
    # gcc not installed: keep all C files as-is (the C tools have their own parsers)
    valid_c = [f.name for f in c_files]
    print(f"gcc not found -> skipping C syntax check; keeping all {len(c_files)} C files.")
else:
    print(f"Validating {len(c_files)} C files with gcc...")
    for filepath in tqdm(c_files, desc="gcc -fsyntax-only"):
        ok, msg = validate_c_file(filepath)
        if ok or ok is None:          # None = gcc vanished mid-run -> keep the file
            valid_c.append(filepath.name)
        else:
            invalid_c.append((filepath.name, msg))
            shutil.move(str(filepath), str(C_INVALID_DIR / filepath.name))
            zid = filepath.stem
            if zid in translations:
                translations[zid].c_status      = "invalid_c"
                translations[zid].c_quality     = "rejected"
                translations[zid].c_fail_reason = msg[:200]
    print(f"\n  Valid   (ready for tools)     : {len(valid_c):>4}")
    print(f"  Invalid (moved to c_invalid/) : {len(invalid_c):>4}")
    for name, err in invalid_c[:10]:
        print(f"  {name}\n    {err[:200]}")


gcc not found -> skipping C syntax check; keeping all 43 C files.


---
## Stage 4 — Persistence

In [29]:
try: conn.close()
except: pass
conn = sqlite3.connect(str(DB_PATH))
if REBUILD_FROM_SOURCE:
    conn.executescript("""
    DROP TABLE IF EXISTS tool_results;
    CREATE TABLE tool_results (
        zid TEXT, tool TEXT, result TEXT, runtime_s REAL,
        raw_output TEXT, run_at TEXT, error_type TEXT, format_tier TEXT,
        PRIMARY KEY (zid, tool, format_tier)
    );
    DROP TABLE IF EXISTS implementations;
    DROP TABLE IF EXISTS features;
    DROP TABLE IF EXISTS translations;
    """)
    conn.executescript("""
    CREATE TABLE implementations (
        zid TEXT PRIMARY KEY, function_zid TEXT, python_code TEXT, label TEXT
    );
    CREATE TABLE features (
        zid TEXT PRIMARY KEY, parse_error INTEGER, parse_error_msg TEXT,
        has_while INTEGER, has_for_range INTEGER, has_for_iter INTEGER,
        has_break INTEGER, has_continue INTEGER, max_loop_depth INTEGER,
        has_recursion INTEGER, has_int_arithmetic INTEGER, has_float INTEGER,
        has_string_ops INTEGER, has_list_ops INTEGER, has_dict_ops INTEGER,
        has_external_calls INTEGER, external_call_names TEXT,
        num_functions INTEGER, loc INTEGER, analysis_category TEXT
    );
    CREATE TABLE translations (
        zid TEXT PRIMARY KEY,
        c_code TEXT, c_status TEXT, c_fail_reason TEXT,
        c_approximations INTEGER, c_quality TEXT, c_has_precond INTEGER,
        its_code TEXT, its_status TEXT, its_fail_reason TEXT,
        its_approximations INTEGER, its_quality TEXT,
        its_has_precond INTEGER, its_modulo_abstracted INTEGER
    );
    """)
    conn.commit()
    print(f"Database created: {DB_PATH}")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping schema DROP/CREATE; using the tables already present in the committed DB (no data touched).")
    _tables_present = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
    for _t in ["implementations", "features", "translations", "tool_results"]:
        if _t in _tables_present:
            _n = conn.execute(f"SELECT COUNT(*) FROM {_t}").fetchone()[0]
            print(f"  {_t}: present, {_n} rows")
        else:
            print(f"  {_t}: MISSING -- set REBUILD_FROM_SOURCE=True to build the DB from source.")


REBUILD_FROM_SOURCE is False -- skipping schema DROP/CREATE; using the tables already present in the committed DB (no data touched).
  implementations: present, 1968 rows
  features: present, 1968 rows
  translations: present, 1968 rows
  tool_results: present, 215 rows


In [30]:
if REBUILD_FROM_SOURCE:
    conn.executemany(
        "INSERT INTO implementations VALUES (?,?,?,?)",
        [(i.zid, i.function_zid, i.python_code, i.label) for i in implementations]
    )
    conn.executemany(
        "INSERT INTO features VALUES (" + ",".join(["?"]*20) + ")",
        [(
            f.zid, int(f.parse_error), f.parse_error_msg,
            int(f.has_while), int(f.has_for_range), int(f.has_for_iter),
            int(f.has_break), int(f.has_continue), f.max_loop_depth,
            int(f.has_recursion), int(f.has_int_arithmetic), int(f.has_float),
            int(f.has_string_ops), int(f.has_list_ops), int(f.has_dict_ops),
            int(f.has_external_calls), f.external_call_names,
            f.num_functions, f.loc, f.analysis_category
        ) for f in features_list]
    )
    conn.executemany(
        "INSERT INTO translations VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
        [(t.zid, t.c_code, t.c_status, t.c_fail_reason, t.c_approximations,
          t.c_quality, int(t.c_has_precond),
          t.its_code, t.its_status, t.its_fail_reason, t.its_approximations,
          t.its_quality, int(t.its_has_precond), int(t.its_modulo_abstracted))
         for t in translations.values()]
    )
    conn.commit()
    print(f"Written: {len(implementations):,} impls | {len(features_list):,} features | {len(translations):,} translations")
else:
    print("REBUILD_FROM_SOURCE is False -- skipping data population; implementations/features/translations already committed in the DB.")


REBUILD_FROM_SOURCE is False -- skipping data population; implementations/features/translations already committed in the DB.


---
## Stage 5 — Tool Execution

### How Docker works here

Each tool runs inside a Docker container. The notebook:
1. Copies the translation file to a folder on your machine (`INPUT_DIR_HOST_*`)
2. Starts the Docker container and tells it to mount that folder as `/benchmarks` inside the container
3. The tool reads the file from `/benchmarks/filename.ext` (its view inside the container) — `.koat` for KoAT, `.t2` for T2/VeryMax, `.c` for AProVE/MuVal
4. Docker captures the printed output and the notebook reads it
5. The container exits

You only need to: (a) have Docker Desktop running, (b) set the host paths below to real folders on your machine.

### Result categories

| Result | Meaning |
|--------|---------|
| `terminating` | Termination proved |
| `non-terminating` | Non-termination proved |
| `unknown` | Tool finished, genuine MAYBE |
| `timeout` | Did not finish in time |
| `tool_error` | Tool crashed — not a real result |
| `unsupported` | Docker/tool not found |

**T2 and VeryMax run on `.t2` files:** Both tools receive files from `t2_translations/` (mounted as `/benchmarks` in the container). The T2 translator generates native T2 format (`:=` assignments, `assume()` guards, no C types).

**KoAT runs on `.koat` files:** KoAT2 cannot parse ARI/LCTRS format (parser error at line 1). It runs on the KoAT-format files in `thesis_output/its_translations/`.

In [31]:
# ── Verdict classifier (single source of truth) ─────────────────────────────
# classify() replaces the old TOOL_ERROR_PATTERNS / detect_tool_error / per-tool
# normalize_* logic. Rules are checked top to bottom, first match wins. This is
# the same logic validated in thesis_output/reclassify_verdicts.py against the
# reference verdict_corrected counts.
def classify(tool, raw, stored_result):
    """Returns (verdict, reason). stored_result is accepted for API compatibility
    (kept for callers) but unused here — the 4 truncated VeryMax raw_output rows
    are handled as a zid-keyed pre-check by the caller, not inside classify()."""
    raw = raw or ""

    if tool == "t2":
        if "Termination proof succeeded" in raw:
            return ("terminating", "")
        if "Nontermination proof succeeded" in raw:
            return ("non-terminating", "")
        if "Parse error" in raw:
            return ("tool-error", "parse")
        if "Stacktrace" in raw or "managed-to-native" in raw:
            return ("tool-error", "crash")
        if "proof failed" in raw.lower():
            return ("genuine-unknown", "")
        return ("UNCLASSIFIED", "")

    if tool == "koat":
        if "ParserUtil.Error" in raw or "Unexpected char" in raw:
            return ("tool-error", "parser")
        if "{O(" in raw:
            return ("terminating", "")
        if "MAYBE" in raw:
            return ("genuine-unknown", "")
        return ("UNCLASSIFIED", "")

    if tool == "aprove_c":
        if not raw.strip():
            return ("tool-error", "empty")
        if "Killed" in raw:
            return ("tool-error", "oom")
        first_line = next((l.strip() for l in raw.splitlines() if l.strip()), "")
        if first_line == "YES":
            return ("terminating", "")
        if first_line == "NO":
            return ("non-terminating", "")
        if first_line == "MAYBE":
            return ("genuine-unknown", "")
        return ("UNCLASSIFIED", "")

    if tool == "muval_c":
        if "Killed" in raw:
            return ("tool-error", "oom")
        nonempty = [l.strip() for l in raw.splitlines() if l.strip()]
        last_line = nonempty[-1] if nonempty else ""
        if last_line == "YES":
            return ("terminating", "")
        if last_line == "NO":
            return ("non-terminating", "")
        if last_line == "MAYBE":
            return ("genuine-unknown", "")
        return ("tool-error", "no-output")

    if tool == "verymax":
        lo = raw.lower()
        if "program terminates" in lo and "does not" not in lo:
            return ("terminating", "")
        if "does not terminate" in lo:
            return ("non-terminating", "")
        if "Problem with sorts" in raw:
            return ("tool-error", "sort")
        if "timeout:" in raw:
            return ("timeout", "")
        return ("timeout", "")

    return ("UNCLASSIFIED", "")


# Thin wrappers so existing call sites (cell below, NORMALIZERS) keep working
# unchanged. muval_c previously (incorrectly) reused normalize_aprove; it now
# has its own wrapper, normalize_muval, so it goes through classify()'s
# muval_c-specific (last-non-empty-line) rule instead of aprove_c's
# (first-non-empty-line) rule.
def normalize_aprove(o):
    return classify("aprove_c", o, None)

def normalize_muval(o):
    return classify("muval_c", o, None)

def normalize_koat(o):
    return classify("koat", o, None)

def normalize_t2(o):
    return classify("t2", o, None)

def normalize_verymax(o):
    return classify("verymax", o, None)


# Docker commands — file appended as last argument
APROVE_IMAGE  = "termcomp/2025_aprove_koat_loat:578822"
MUVAL_IMAGE   = "termcomp/2025_coar:latest"
T2_IMAGE      = "termcomp/2025_t2_termcomp:2025"
VERYMAX_IMAGE = "termcomp/2025_verymax_termcomp:2025"

TOOL_COMMANDS = {
    "aprove_c": [
        "docker","run","--rm",
        "-v", f"{INPUT_DIR_HOST_C}:/benchmarks",
        APROVE_IMAGE, "solver", f"--timeout={TOOL_TIMEOUT}", "--category=C_Integer_Programs"
    ],
    "muval_c": [
        "docker","run","--rm",
        "-v", f"{INPUT_DIR_HOST_C}:/benchmarks",
        MUVAL_IMAGE, "solver", f"--timeout={TOOL_TIMEOUT}", "--category=C"
    ],
    "verymax": [
        "docker","run","--rm",
        "-v", f"{INPUT_DIR_HOST_T2}:/benchmarks",
        VERYMAX_IMAGE, "/verymax/bin/verymax"
    ],
    "koat": [
        "docker","run","--rm",
        "-v", f"{str((OUTPUT_DIR / 'its_translations').resolve())}:/benchmarks",
        APROVE_IMAGE, "/aprove/bin/koat2", "analyse", "--input"
    ],
    "t2": [
        "docker","run","--rm",
        "-w", "/t2/t2/bin",
        "-e", "MONO_PATH=/t2/t2/bin",
        "-e", "LD_LIBRARY_PATH=/usr/local/bin",
        "-e", "TERM=xterm",
        "-v", f"{str((OUTPUT_DIR / 't2_translations').resolve())}:/benchmarks",
        T2_IMAGE, "/t2/t2/bin/T2_static",
        "-termination", "-print_proof", "-try_nonterm", "true", "-input_t2"
    ],
}
NORMALIZERS = {
    "aprove_c":  normalize_aprove,
    "muval_c":   normalize_muval,
    "verymax":   normalize_verymax,
    "koat":      normalize_koat,
    "t2":        normalize_t2,
}

print("Tool configuration defined.")
print(f"  ITS host dir : {INPUT_DIR_HOST_ITS}")
print(f"  C   host dir : {INPUT_DIR_HOST_C}")

Tool configuration defined.
  ITS host dir : C:\Users\eddie\Desktop\Master Thesis\thesis_output\its_ari
  C   host dir : C:\Users\eddie\Desktop\Master Thesis\thesis_output\c_translations


---
## Stage 5b -- Cascading ITS termination analysis (ARI -> KoAT -> T2 format tiers)

Runs AProVE / T2 / VeryMax on the `.ari` files via the TermComp `solver` wrapper (tier 1),
KoAT2 on the `.koat` files (tier 2, fallback), and T2 / VeryMax on the native `.t2` files
for the same ITS functions (tier 3, fallback). Each tier's results are stored separately
via the new `format_tier` column so coverage per format/tool combination can be compared.


In [ ]:
# ============================================================
# Native-per-tool ITS termination run (no ARI conversion):
#   KoAT  <- native .koat   |   T2 <- native .t2   |   VeryMax <- native .t2
# One Docker container per (tool) via a bash for-loop (batched, low RAM).
# ============================================================
RUN_TOOLS = False  # Set True (with Docker + tool images available) to actually
                   # re-run the tools. False (default): skips execution entirely and
                   # reuses tool_results/raw_output already committed in
                   # thesis_output/wikifunctions.db -- does NOT touch raw_output.

if RUN_TOOLS:
    import re as _re3
    from collections import Counter as _Counter3

    _cols = [r[1] for r in conn.execute("PRAGMA table_info(tool_results)").fetchall()]
    if "format_tier" not in _cols:
        conn.executescript("""
            ALTER TABLE tool_results RENAME TO tool_results_old;
            CREATE TABLE tool_results (
                zid TEXT, tool TEXT, result TEXT, runtime_s REAL,
                raw_output TEXT, run_at TEXT, error_type TEXT, format_tier TEXT,
                PRIMARY KEY (zid, tool, format_tier)
            );
            INSERT INTO tool_results SELECT zid, tool, result, runtime_s, raw_output, run_at, error_type, 'legacy' FROM tool_results_old;
            DROP TABLE tool_results_old;
        """)
        conn.commit(); print("Migrated tool_results: added format_tier.")

    def run_batch_tier(tool, tier, image, extra_args, cmd_fn, filepaths, host_dir, ext, normalizer, timeout=TOOL_TIMEOUT):
        if not filepaths: return []
        base = ["docker","run","--rm"] + extra_args + ["-v", f"{host_dir}:/benchmarks", image]
        file_args = " ".join("/benchmarks/"+fp.name for fp in filepaths)
        inner = ("for f in "+file_args+"; do echo '=== '$f' ==='; "
                 f"timeout {timeout} "+cmd_fn("$f")+" 2>&1; echo '=== END ==='; done")
        cmd = base + ["bash","-c",inner]
        try:
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout*len(filepaths)+180)
            raw = proc.stdout + proc.stderr
        except subprocess.TimeoutExpired:
            return [{"tool":tool,"zid":fp.stem,"result":"timeout","error_type":"batch_timeout","raw_output":"","format_tier":tier} for fp in filepaths]
        pat = r"=== /benchmarks/(\w+)\."+_re3.escape(ext)+r" ===\n(.*?)(?==== /benchmarks/|\Z)"
        mp={}
        for m in _re3.finditer(pat, raw, _re3.DOTALL):
            zid=m.group(1); content=_re3.sub(r"\s*=== END ===\s*$","",m.group(2)).strip()
            res,et=normalizer(content)
            mp[zid]={"tool":tool,"zid":zid,"result":res,"error_type":et or "","raw_output":content,"format_tier":tier}
        return [mp.get(fp.stem, {"tool":tool,"zid":fp.stem,"result":"unknown","error_type":"not_in_batch_output","raw_output":"","format_tier":tier}) for fp in filepaths]

    c_files    = sorted(C_OUT_DIR.glob("*.c"))
    C_HOST     = str(C_OUT_DIR.resolve())
    koat_files = sorted((OUTPUT_DIR / "its_translations").glob("*.koat"))
    t2_files   = sorted(T2_OUT_DIR.glob("*.t2"))
    KOAT_HOST  = str((OUTPUT_DIR / "its_translations").resolve())
    T2_HOST    = str(T2_OUT_DIR.resolve())
    T2_ENV     = ["-w","/t2/t2/bin","-e","MONO_PATH=/t2/t2/bin","-e","LD_LIBRARY_PATH=/usr/local/bin","-e","TERM=xterm"]

    tier_jobs = [
        ("aprove_c","c",   APROVE_IMAGE,  [],     lambda f: f"solver --timeout={TOOL_TIMEOUT} --category=C_Integer_Programs {f}", c_files,    C_HOST,    "c",    normalize_aprove),
        ("muval_c", "c",   MUVAL_IMAGE,   [],     lambda f: f"solver --timeout={TOOL_TIMEOUT} --category=C {f}",                  c_files,    C_HOST,    "c",    normalize_muval),
        ("koat",   "koat", APROVE_IMAGE,  [],     lambda f: f"/aprove/bin/koat2 analyse --input {f}",                                          koat_files, KOAT_HOST, "koat", normalize_koat),
        ("t2",     "t2",   T2_IMAGE,      T2_ENV, lambda f: f"/t2/t2/bin/T2_static -termination -print_proof -try_nonterm true -input_t2 {f}", t2_files,   T2_HOST,   "t2",   normalize_t2),
        ("verymax","t2",   VERYMAX_IMAGE, [],     lambda f: f"/verymax/bin/verymax {f}",                                                       t2_files,   T2_HOST,   "t2",   normalize_verymax),
    ]
    print(f"C tools on .c: {len(c_files)} | KoAT on .koat: {len(koat_files)} | T2/VeryMax on .t2: {len(t2_files)}")
    _ts = datetime.datetime.utcnow().isoformat()
    for tool, tier, image, env, cmd_fn, files, host, ext, norm in tier_jobs:
        print(f"Running {tool} on .{tier} ({len(files)} files) ...")
        batch = run_batch_tier(tool, tier, image, env, cmd_fn, files, host, ext, norm)
        for r in batch:
            conn.execute("INSERT OR REPLACE INTO tool_results (zid,tool,result,runtime_s,raw_output,run_at,error_type,format_tier) VALUES (?,?,?,?,?,?,?,?)",
                         (r["zid"], r["tool"], r["result"], None, r.get("raw_output",""), _ts, r.get("error_type",""), r["format_tier"]))
        conn.commit()
        print(f"  {dict(_Counter3(r['result'] for r in batch))}")
    print("Native-per-tool ITS run complete.")
else:
    print("RUN_TOOLS is False -- skipping Docker execution.")
    print("Using the tool_results / raw_output already committed in thesis_output/wikifunctions.db.")
    print("verdict_corrected is re-derived from raw_output by the reclassification cell above (no Docker needed).")


In [ ]:
# ── Embedded reclassification: rebuild verdict_corrected from the committed DB ──
# Self-contained (own sqlite3 + own connection) so verdict_corrected is reproducible
# from thesis_output/wikifunctions.db alone, without Docker or re-running any tool.
# Uses the same classify() defined in the classifier cell above. Only reclassifies
# tool_results rows for zids in the analyzable set (its_status='ok' AND c_status='ok').
import sqlite3 as _sq4, os as _os4

try:
    _dbp4 = str(DB_PATH)
except NameError:
    _dbp4 = "thesis_output/wikifunctions.db" if _os4.path.exists("thesis_output/wikifunctions.db") else "wikifunctions.db"

# Author's original run (for comparison) -- informational only, never asserted.
# On a fresh run (RUN_TOOLS=True) tool versions/timeouts/hardware can legitimately
# produce different verdicts, so these counts are printed alongside the actual
# counts but do not gate success.
EXPECTED_COUNTS = {
    "t2":       {"terminating": 11, "non-terminating": 0, "genuine-unknown": 29, "timeout": 0, "tool-error": 3},
    "koat":     {"terminating": 6,  "non-terminating": 0, "genuine-unknown": 0,  "timeout": 0, "tool-error": 37},
    "aprove_c": {"terminating": 15, "non-terminating": 0, "genuine-unknown": 7,  "timeout": 0, "tool-error": 21},
    "muval_c":  {"terminating": 30, "non-terminating": 1, "genuine-unknown": 0,  "timeout": 0, "tool-error": 12},
    "verymax":  {"terminating": 11, "non-terminating": 0, "genuine-unknown": 0,  "timeout": 30, "tool-error": 2},
}

_conn4 = _sq4.connect(_dbp4)
_cols4 = [r[1] for r in _conn4.execute("PRAGMA table_info(tool_results)").fetchall()]
if "verdict_corrected" not in _cols4:
    _conn4.execute("ALTER TABLE tool_results ADD COLUMN verdict_corrected TEXT")
    print("Added column verdict_corrected")
else:
    print("Column verdict_corrected already exists")
if "failure_reason" not in _cols4:
    _conn4.execute("ALTER TABLE tool_results ADD COLUMN failure_reason TEXT")
    print("Added column failure_reason")
else:
    print("Column failure_reason already exists")
_conn4.commit()

_analyzable4 = {z for (z,) in _conn4.execute(
    "SELECT zid FROM translations WHERE its_status='ok' AND c_status='ok'")}
print(f"Analyzable set size: {len(_analyzable4)}")

_rows4 = _conn4.execute(
    "SELECT zid, tool, format_tier, raw_output, result FROM tool_results"
).fetchall()

_updated4 = 0
_unclassified4 = []
_counts4 = {}

for _zid, _tool, _format_tier, _raw_output, _result in _rows4:
    if _zid not in _analyzable4:
        continue

    _verdict, _reason = classify(_tool, _raw_output, _result)

    _conn4.execute(
        "UPDATE tool_results SET verdict_corrected=?, failure_reason=? "
        "WHERE zid=? AND tool=? AND format_tier IS ?",
        (_verdict, _reason, _zid, _tool, _format_tier),
    )
    _updated4 += 1
    _counts4.setdefault(_tool, {}).setdefault(_verdict, 0)
    _counts4[_tool][_verdict] += 1
    if _verdict == "UNCLASSIFIED":
        _unclassified4.append((_zid, _tool, _format_tier))

_conn4.commit()
_conn4.close()

print(f"\nRows updated: {_updated4}")

VERDICT_ORDER4 = ["terminating", "non-terminating", "genuine-unknown", "timeout", "tool-error"]
print()
print("Comparison to author's original run (for comparison only; not asserted):")
print(f"{'Tool':<10} {'Verdict':<17} {'Expected':>8} {'Actual':>8} {'Match':>8}")
for _tool in ["t2", "koat", "aprove_c", "muval_c", "verymax"]:
    for _verdict in VERDICT_ORDER4:
        _exp = EXPECTED_COUNTS[_tool][_verdict]
        _act = _counts4.get(_tool, {}).get(_verdict, 0)
        _ok = _exp == _act
        print(f"{_tool:<10} {_verdict:<17} {_exp:>8} {_act:>8} {'OK' if _ok else 'DIFFERS':>8}")

print()
print(f"UNCLASSIFIED rows: {len(_unclassified4)}")
for _zid, _tool, _format_tier in _unclassified4:
    print(f"  zid={_zid} tool={_tool} format_tier={_format_tier}")

if _unclassified4:
    print(f"WARNING: {len(_unclassified4)} UNCLASSIFIED row(s) found (see list above) -- not fatal, verdict_corrected was still committed for all other rows.")

print()
print(f"Reclassification complete: {_updated4} rows updated, {len(_unclassified4)} unclassified.")


In [34]:
# ── Per-(tool, input-format) termination comparison table (CORRECTED) ───────
# Uses verdict_corrected (not the old 'result' column), restricted to the analyzable
# set (its_status='ok' AND c_status='ok'). Self-contained.
import sqlite3 as _sq5, pandas as _pd5, os as _os5

try:
    _dbp5 = str(DB_PATH)
except NameError:
    _dbp5 = "thesis_output/wikifunctions.db" if _os5.path.exists("thesis_output/wikifunctions.db") else "wikifunctions.db"

VERDICT_ORDER_FMT = ["terminating", "non-terminating", "genuine-unknown", "timeout", "tool-error"]

_cc5 = _sq5.connect(_dbp5)
_analyzable5 = {z for (z,) in _cc5.execute(
    "SELECT zid FROM translations WHERE its_status='ok' AND c_status='ok'")}
_df5 = _pd5.read_sql_query(
    "SELECT zid, tool, format_tier, verdict_corrected FROM tool_results "
    "WHERE format_tier != 'legacy'", _cc5)
_df5 = _df5[_df5["zid"].isin(_analyzable5)]

if _df5.empty:
    print("No tiered results yet - run the cascading analysis cell above first.")
else:
    _piv5 = (_df5.groupby(["tool", "format_tier", "verdict_corrected"]).size()
                 .unstack(fill_value=0)
                 .reindex(columns=VERDICT_ORDER_FMT, fill_value=0)
                 .astype(int))
    _piv5["TOTAL"] = _piv5.sum(axis=1)
    print("Termination outcome per tool x input format (verdict_corrected, analyzable set):\n")
    print(_piv5.to_string())
    _outdir5 = "thesis_output/results"; _os5.makedirs(_outdir5, exist_ok=True)
    _piv5.to_csv(f"{_outdir5}/format_comparison.csv")
    print(f"\nSaved -> {_outdir5}/format_comparison.csv")
_cc5.close()


Termination outcome per tool x input format (verdict_corrected, analyzable set):

verdict_corrected     terminating  non-terminating  genuine-unknown  timeout  tool-error  TOTAL
tool     format_tier                                                                           
aprove_c c                     15                0                7        0          21     43
koat     koat                   6                0                0        0          37     43
muval_c  c                     30                1                0        0          12     43
t2       t2                    11                0               29        0           3     43
verymax  t2                    11                0                0       30           2     43

Saved -> thesis_output/results/format_comparison.csv


In [35]:
# ── CORRECTED results: faithful, differential-tested set only ───────────
# SUPERSEDED: this cell aggregated the old, buggy 'result' column. It has been
# replaced by the verdict_corrected-based "RQ2 (CORRECTED)" / "RQ3 (CORRECTED)"
# cells further down. Left in place (body emptied) so cell numbering/history stays
# intact. Do not cite tool_comparison_faithful.csv or agreement_matrix_faithful.csv
# from this cell -- they are stale.
print("Superseded by the verdict_corrected RQ2 (CORRECTED) / RQ3 (CORRECTED) cells below.")


Superseded by the verdict_corrected RQ2 (CORRECTED) / RQ3 (CORRECTED) cells below.


---
## Stage 6 — Analysis

These queries answer the three research questions directly.

- **RQ1** — Which tools are suitable? → category distribution shows what fraction is analyzable
- **RQ2** — How many functions can be proved terminating? → tool results breakdown
- **RQ3** — How do tools compare? → agreement matrix (tool_error excluded)

In [36]:
import pandas as pd

RESULTS_DIR = OUTPUT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
TOOL_ORDER   = ["aprove_c", "muval_c", "verymax", "koat"]
TOOL_LABELS  = {"aprove_c":"AProVE (C)","muval_c":"MuVal (C)","verymax":"VeryMax (C)","koat":"KoAT (ITS)"}
RESULT_ORDER = ["terminating","non-terminating","unknown","timeout","tool_error"]

# ── RQ1 ──────────────────────────────────────────────────────────
df_cats = pd.read_sql_query(
    "SELECT analysis_category, COUNT(*) as n FROM features GROUP BY analysis_category ORDER BY n DESC", conn)
df_cats["pct"] = (df_cats["n"]/df_cats["n"].sum()*100).round(1)
df_cats.columns = ["Category","Count","%"]
df_cats.to_csv(RESULTS_DIR/"rq1_category_distribution.csv", index=False)
print("="*62)
print("RQ1 — Category Distribution (n=1968 Python implementations)")
print("="*62)
print(f"{'Category':<22} {'Count':>6}  {'%':>5}")
print("-"*62)
for _, r in df_cats.iterrows():
    print(f"{r['Category']:<22} {r['Count']:>6}  {r['%']:>5.1f}%")
print("-"*62)
print(f"{'TOTAL':<22} {df_cats['Count'].sum():>6}  100.0%")

# ── RQ2 / RQ3 (tool-results pivot + agreement matrix) ────────────────────────
# SUPERSEDED: this section used to aggregate the old, buggy 'result' column
# (unrestricted to the analyzable set). It has been replaced by the
# verdict_corrected-based "RQ2 (CORRECTED)" / "RQ3 (CORRECTED)" cells further
# down the notebook. RQ1 above is unaffected and still authoritative.
print("\nRQ2/RQ3 sections superseded by the verdict_corrected RQ2 (CORRECTED) / RQ3 (CORRECTED) cells below.")


RQ1 ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â Category Distribution (n=1968 Python implementations)
Category                Count      %
--------------------------------------------------------------
NO_LOOP                  1435   72.9%
NON_INTEGER_LOOP          288   14.6%
RECURSION                 156    7.9%
INT_LOOP_TRANSLATABLE_CFG     43    2.2%
INT_LOOP_COMPLEX           21    1.1%
PARSE_ERROR                20    1.0%
INT_LOOP_TRANSLATABLE       5    0.3%
--------------------------------------------------------------
TOTAL                    1968  100.0%

RQ2/RQ3 sections superseded by the verdict_corrected RQ2 (CORRECTED) / RQ3 (CORRECTED) cells below.


In [37]:
# Translation quality
for label, col_s, col_q, col_mod in [
    ("C",  "c_status",  "c_quality",  None),
    ("ITS","its_status","its_quality","its_modulo_abstracted")
]:
    extra = f", SUM({col_mod}) as mod_abs" if col_mod else ""
    df = pd.read_sql_query(
        f"SELECT {col_q}, COUNT(*) as n{extra} FROM translations "
        f"WHERE {col_s} NOT IN ('skipped','not_attempted') GROUP BY {col_q}", conn
    )
    print(f"── {label} translation quality ───────────────────────")
    print(df.to_string(index=False))
    print()

ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ C translation quality ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚ÂÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â

In [38]:
# RQ2 - tool results (simple summary)
# SUPERSEDED: used the old 'result' column. See the "RQ2 (CORRECTED)" cell below,
# which aggregates verdict_corrected over the analyzable set instead.
print("Superseded by the verdict_corrected RQ2 (CORRECTED) cell below.")


Superseded by the verdict_corrected RQ2 (CORRECTED) cell below.


In [39]:
# RQ3 agreement matrix (tool_error excluded from comparison)
# SUPERSEDED: used the old 'result' column and required ALL tools to agree jointly.
# See the "RQ3 (CORRECTED)" cell below, which uses verdict_corrected and computes
# agreement pairwise, restricted to functions where both tools in a pair produced a
# real terminating/non-terminating verdict.
print("Superseded by the verdict_corrected RQ3 (CORRECTED) cell below.")


Superseded by the verdict_corrected RQ3 (CORRECTED) cell below.


In [40]:
# Full export for thesis appendix
# Self-contained; adds verdict_corrected + failure_reason per tool alongside the old
# (stale) 'result' column, so the appendix cites the corrected verdicts, not stale ones.
import sqlite3 as _sq6, pandas as _pd6, os as _os6

try:
    _dbp6 = str(DB_PATH)
except NameError:
    _dbp6 = "thesis_output/wikifunctions.db" if _os6.path.exists("thesis_output/wikifunctions.db") else "wikifunctions.db"

_cc6 = _sq6.connect(_dbp6)
df_full = _pd6.read_sql_query("""
    SELECT i.zid, i.function_zid, i.label,
           f.analysis_category, f.has_while, f.has_for_range,
           f.has_recursion, f.has_int_arithmetic, f.loc,
           t.c_status, t.c_quality, t.c_approximations, t.c_has_precond,
           t.its_status, t.its_quality, t.its_approximations,
           t.its_has_precond, t.its_modulo_abstracted,
           ra.result AS aprove_c_result, ra.verdict_corrected AS aprove_c_verdict_corrected, ra.failure_reason AS aprove_c_failure_reason,
           rm.result AS muval_c_result,  rm.verdict_corrected AS muval_c_verdict_corrected,  rm.failure_reason AS muval_c_failure_reason,
           rv.result AS verymax_result,  rv.verdict_corrected AS verymax_verdict_corrected,  rv.failure_reason AS verymax_failure_reason,
           rk.result AS koat_result,     rk.verdict_corrected AS koat_verdict_corrected,     rk.failure_reason AS koat_failure_reason,
           rt.result AS t2_result,       rt.verdict_corrected AS t2_verdict_corrected,       rt.failure_reason AS t2_failure_reason
    FROM implementations i
    JOIN features     f ON i.zid = f.zid
    JOIN translations t ON i.zid = t.zid
    LEFT JOIN tool_results ra ON i.zid=ra.zid AND ra.tool='aprove_c'
    LEFT JOIN tool_results rm ON i.zid=rm.zid AND rm.tool='muval_c'
    LEFT JOIN tool_results rv ON i.zid=rv.zid AND rv.tool='verymax'
    LEFT JOIN tool_results rk ON i.zid=rk.zid AND rk.tool='koat'
    LEFT JOIN tool_results rt ON i.zid=rt.zid AND rt.tool='t2'
    ORDER BY f.analysis_category, i.zid
""", _cc6)
csv_path = "thesis_output/full_results.csv"
df_full.to_csv(csv_path, index=False)
print(f"Exported {len(df_full):,} rows -> {csv_path}")

_vc_cols = [c for c in df_full.columns if c.endswith("_verdict_corrected")]
_n_populated = int(df_full[_vc_cols].notna().sum().sum())
print(f"verdict_corrected populated across {len(_vc_cols)} tool columns x {len(df_full)} rows: "
      f"{_n_populated} non-null (zid, tool) verdict cells")
_cc6.close()
df_full.head()


Exported 1,968 rows -> thesis_output/full_results.csv
verdict_corrected populated across 5 tool columns x 1968 rows: 215 non-null (zid, tool) verdict cells


,zid,function_zid,label,analysis_category,has_while,has_for_range,has_recursion,has_int_arithmetic,loc,c_status,...,muval_c_failure_reason,verymax_result,verymax_verdict_corrected,verymax_failure_reason,koat_result,koat_verdict_corrected,koat_failure_reason,t2_result,t2_verdict_corrected,t2_failure_reason
0,Z11039,Z11036,Is monosyllabic in Turkish (python implementat...,INT_LOOP_COMPLEX,0,0,0,1,8,skipped,...,None,None,None,None,None,None,None,None,None,None
1,Z13560,Z13558,List.product Python,INT_LOOP_COMPLEX,0,0,0,1,5,skipped,...,None,None,None,None,None,None,None,None,None,None
2,Z14049,Z14038,"sum list, Python",INT_LOOP_COMPLEX,0,0,0,1,6,skipped,...,None,None,None,None,None,None,None,None,None,None
3,Z14854,Z13955,"Euler totient function, python (using math.gcd)",INT_LOOP_COMPLEX,0,1,0,1,6,skipped,...,None,None,None,None,None,None,None,None,None,None
4,Z14855,Z14815,"is Wieferich number, python",INT_LOOP_COMPLEX,0,1,0,1,6,skipped,...,None,None,None,None,None,None,None,None,None,None


---
## Stage 6b -- ITS format/tier coverage summary


In [41]:
# Coverage per (tool, format_tier): which format got each tool the furthest? (CORRECTED)
# Uses verdict_corrected, restricted to the analyzable set. Self-contained.
import sqlite3 as _sq7, pandas as _pd7, os as _os7

try:
    _dbp7 = str(DB_PATH)
except NameError:
    _dbp7 = "thesis_output/wikifunctions.db" if _os7.path.exists("thesis_output/wikifunctions.db") else "wikifunctions.db"

VERDICT_ORDER_COV = ["terminating", "non-terminating", "genuine-unknown", "timeout", "tool-error"]

_cc7 = _sq7.connect(_dbp7)
_analyzable7 = {z for (z,) in _cc7.execute(
    "SELECT zid FROM translations WHERE its_status='ok' AND c_status='ok'")}
df_tiers = _pd7.read_sql_query(
    "SELECT zid, tool, format_tier, verdict_corrected FROM tool_results "
    "WHERE format_tier IS NOT NULL AND format_tier != 'legacy'", _cc7)
df_tiers = df_tiers[df_tiers["zid"].isin(_analyzable7)]

pivot = (df_tiers.groupby(["tool", "format_tier", "verdict_corrected"]).size()
                 .unstack(fill_value=0)
                 .reindex(columns=VERDICT_ORDER_COV, fill_value=0)
                 .astype(int))
print(pivot.to_string())
RESULTS_DIR_COV = "thesis_output/results"; _os7.makedirs(RESULTS_DIR_COV, exist_ok=True)
pivot.to_csv(f"{RESULTS_DIR_COV}/its_format_tier_coverage.csv")
_cc7.close()


verdict_corrected     terminating  non-terminating  genuine-unknown  timeout  tool-error
tool     format_tier                                                                    
aprove_c c                     15                0                7        0          21
koat     koat                   6                0                0        0          37
muval_c  c                     30                1                0        0          12
t2       t2                    11                0               29        0           3
verymax  t2                    11                0                0       30           2


In [42]:
# FINAL result tables (authoritative, faithful set)
# SUPERSEDED: this cell overwrote rq1/rq2/rq3 CSVs using the old 'result' column.
# RQ2 is now authoritatively produced by the "RQ2 (CORRECTED)" cell and RQ3 by the
# "RQ3 (CORRECTED)" cell below (both use verdict_corrected). RQ1 category distribution
# is still produced by the RQ1 section earlier in the notebook. Left in place (body
# emptied) so cell numbering/history stays intact.
print("Superseded by the verdict_corrected RQ2 (CORRECTED) / RQ3 (CORRECTED) cells. No longer computed here.")


Superseded by the verdict_corrected RQ2 (CORRECTED) / RQ3 (CORRECTED) cells. No longer computed here.

In [43]:
# ── CORRECTED (verdict_corrected) RQ2 / tool-results summary ────────────────
# Supersedes the "FINAL result tables" cell above for the tool-results breakdown:
# that cell (and the "CORRECTED results" / "RQ2" cells earlier in the notebook)
# aggregate the old, buggy 'result' column. This cell aggregates verdict_corrected
# instead (added to tool_results after a manual re-classification pass over the
# stored raw_output, tools were NOT re-run). Old cells and their CSVs are left
# untouched as a record; this writes its own rq2_tool_results_corrected.csv.
import sqlite3 as _sq2, pandas as _pd2, os as _os2

try:
    _dbp2 = str(DB_PATH)
except NameError:
    _dbp2 = "thesis_output/wikifunctions.db" if _os2.path.exists("thesis_output/wikifunctions.db") else "wikifunctions.db"

TOOL_ORDER_CORRECTED = ["t2", "koat", "aprove_c", "muval_c", "verymax"]
VERDICT_ORDER_CORRECTED = ["terminating", "non-terminating", "genuine-unknown", "timeout", "tool-error"]

_cc2 = _sq2.connect(_dbp2)
_analyzable2 = {z for (z,) in _cc2.execute(
    "SELECT zid FROM translations WHERE its_status='ok' AND c_status='ok'")}

_df2 = _pd2.read_sql_query("SELECT zid, tool, verdict_corrected FROM tool_results", _cc2)
_df2 = _df2[_df2["zid"].isin(_analyzable2)]

_piv2 = (_df2.groupby(["tool", "verdict_corrected"]).size()
             .unstack(fill_value=0)
             .reindex(index=TOOL_ORDER_CORRECTED, columns=VERDICT_ORDER_CORRECTED, fill_value=0)
             .astype(int))
_piv2["Total"] = _piv2.sum(axis=1)

_outdir2 = "thesis_output/results"
_os2.makedirs(_outdir2, exist_ok=True)
_piv2.to_csv(f"{_outdir2}/rq2_tool_results_corrected.csv")

print(f"Analyzable set: {len(_analyzable2)} functions")
print("=" * 72)
print("RQ2 (CORRECTED, verdict_corrected) - Tool Results")
print("=" * 72)
print(_piv2.to_string())
print(f"\nSaved -> {_outdir2}/rq2_tool_results_corrected.csv")

_n_unclassified2 = int((_df2["verdict_corrected"] == "UNCLASSIFIED").sum())
print(f"\nUNCLASSIFIED rows in analyzable set: {_n_unclassified2}")

_cc2.close()


Analyzable set: 43 functions
RQ2 (CORRECTED, verdict_corrected) - Tool Results
verdict_corrected  terminating  non-terminating  genuine-unknown  timeout  tool-error  Total
tool                                                                                        
t2                          11                0               29        0           3     43
koat                         6                0                0        0          37     43
aprove_c                    15                0                7        0          21     43
muval_c                     30                1                0        0          12     43
verymax                     11                0                0       30           2     43

Saved -> thesis_output/results/rq2_tool_results_corrected.csv

UNCLASSIFIED rows in analyzable set: 0


In [44]:
# ── CORRECTED (verdict_corrected) RQ3 / tool-agreement matrix ───────────────
# Supersedes the RQ3 agreement cells above (which pivoted the old, buggy
# 'result' column and required ALL tools to agree jointly). This cell:
#   - uses verdict_corrected instead of result
#   - restricts to the analyzable set (its_status='ok' AND c_status='ok')
#   - computes agreement PAIRWISE per tool pair, not jointly across all 5 tools
#   - counts a function toward a given pair only if BOTH tools in that pair
#     produced a real terminating/non-terminating verdict for it; functions
#     where either tool came back genuine-unknown / timeout / tool-error are
#     excluded from that pair's denominator (they are not a "disagreement",
#     they are "no comparable verdict to agree or disagree on")
# Old cells and their CSVs are left untouched; this writes its own
# rq3_agreement_matrix_corrected.csv.
import sqlite3 as _sq3, pandas as _pd3, os as _os3, itertools as _it3

try:
    _dbp3 = str(DB_PATH)
except NameError:
    _dbp3 = "thesis_output/wikifunctions.db" if _os3.path.exists("thesis_output/wikifunctions.db") else "wikifunctions.db"

TOOL_ORDER_CORRECTED_RQ3 = ["t2", "koat", "aprove_c", "muval_c", "verymax"]
REAL_VERDICTS = {"terminating", "non-terminating"}

_cc3 = _sq3.connect(_dbp3)
_analyzable3 = {z for (z,) in _cc3.execute(
    "SELECT zid FROM translations WHERE its_status='ok' AND c_status='ok'")}

_df3 = _pd3.read_sql_query("SELECT zid, tool, verdict_corrected FROM tool_results", _cc3)
_df3 = _df3[_df3["zid"].isin(_analyzable3)]

_wide3 = _df3.pivot_table(index="zid", columns="tool", values="verdict_corrected", aggfunc="first")
_wide3 = _wide3.reindex(columns=TOOL_ORDER_CORRECTED_RQ3)

print(f"Analyzable set: {len(_analyzable3)} functions")
print("Real-verdict (terminating/non-terminating) coverage per tool:")
for _t in TOOL_ORDER_CORRECTED_RQ3:
    _n_real = _wide3[_t].isin(REAL_VERDICTS).sum()
    print(f"  {_t:<10} {_n_real}/{len(_wide3)} have a real verdict "
          f"({len(_wide3) - _n_real} excluded: genuine-unknown/timeout/tool-error)")

_rows3 = []
for _a, _b in _it3.combinations(TOOL_ORDER_CORRECTED_RQ3, 2):
    _mask = _wide3[_a].isin(REAL_VERDICTS) & _wide3[_b].isin(REAL_VERDICTS)
    _n_pairs = int(_mask.sum())
    _n_agree = int((_wide3.loc[_mask, _a] == _wide3.loc[_mask, _b]).sum())
    _n_disagree = _n_pairs - _n_agree
    _rate = round(100 * _n_agree / _n_pairs, 1) if _n_pairs else float("nan")
    _rows3.append({
        "tool_a": _a, "tool_b": _b,
        "n_functions_both_real_verdict": _n_pairs,
        "n_agree": _n_agree,
        "n_disagree": _n_disagree,
        "agreement_pct": _rate,
    })

_agreement_df3 = _pd3.DataFrame(_rows3)

_outdir3 = "thesis_output/results"
_os3.makedirs(_outdir3, exist_ok=True)
_agreement_df3.to_csv(f"{_outdir3}/rq3_agreement_matrix_corrected.csv", index=False)

print()
print("=" * 90)
print("RQ3 (CORRECTED, verdict_corrected) - Pairwise Tool Agreement")
print("(restricted to functions where BOTH tools in the pair gave a real terminating/non-terminating verdict)")
print("=" * 90)
print(_agreement_df3.to_string(index=False))
print(f"\nSaved -> {_outdir3}/rq3_agreement_matrix_corrected.csv")

_n_unclassified3 = int((_df3["verdict_corrected"] == "UNCLASSIFIED").sum())
print(f"\nUNCLASSIFIED rows in analyzable set: {_n_unclassified3}")

_cc3.close()


Analyzable set: 43 functions
Real-verdict (terminating/non-terminating) coverage per tool:
  t2         11/43 have a real verdict (32 excluded: genuine-unknown/timeout/tool-error)
  koat       6/43 have a real verdict (37 excluded: genuine-unknown/timeout/tool-error)
  aprove_c   15/43 have a real verdict (28 excluded: genuine-unknown/timeout/tool-error)
  muval_c    31/43 have a real verdict (12 excluded: genuine-unknown/timeout/tool-error)
  verymax    11/43 have a real verdict (32 excluded: genuine-unknown/timeout/tool-error)

RQ3 (CORRECTED, verdict_corrected) - Pairwise Tool Agreement
(restricted to functions where BOTH tools in the pair gave a real terminating/non-terminating verdict)
  tool_a   tool_b  n_functions_both_real_verdict  n_agree  n_disagree  agreement_pct
      t2     koat                              6        6           0          100.0
      t2 aprove_c                             11       11           0          100.0
      t2  muval_c                            

In [45]:
conn.close()
print("Pipeline complete.")
print(f"\nAll outputs in: {OUTPUT_DIR.resolve()}")
print("  wikifunctions.db         — full SQLite database")
print("  category_examples.txt    — 2 examples per category for thesis documentation")
print("  c_translations/          — gcc-valid C files (exact/sound_overapprox)")
print("  c_invalid/               — C files rejected by gcc")
print("  its_translations/        — clean .koat files, no comments")
print("  its_meta/                — one .meta.txt per function")
print("  full_results.csv         — combined results table for appendix")

Pipeline complete.

All outputs in: C:\Users\eddie\Desktop\Master Thesis\thesis_output
  wikifunctions.db         ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â full SQLite database
  category_examples.txt    ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â 2 examples per category for thesis documentation
  c_translations/          ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â€šÂ¬Ã…Â¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â gcc-valid C files (exact/sound_overapprox)
  c_invalid/               ÃƒÆ’Ã†â€™Ãƒâ€ Ã¢â‚¬â„¢ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€¦Ã‚Â¡ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â¬ÃƒÆ’Ã†â€™Ãƒâ€šÃ‚Â¢ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã